In [ ]:
# ============================================================
# CELL 1 — DEPENDENCY INSTALLATION (REVISED)
# YOLO26s + DINOv3 + STANDARD MAE
# ============================================================

import sys
import subprocess

packages = [
    "ultralytics==8.4.182",
    "transformers>=4.56.0,<5",
    "huggingface_hub>=0.34,<1.0",
    "Pillow>=10.0,<13.0",
    "lime==0.2.0.1",
    "thop",
    "opencv-python-headless",
    "scikit-image",
    "pyyaml",
    "tqdm"
]

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "-q", *packages],
    check=True
)

print("Required project packages installed.")

In [ ]:
# ============================================================
# CELL 1A — SETUP: IMPORTS, PATHS, CONSTANTS (RESTORED — CHECK THESE VALUES)
# The uploaded notebook used W, DS, CKPT, LOGS, RUNS, EV, FIG, SEED, NC, SC,
# TAG, DEVICE, ... but had no cell that defines them. This cell rebuilds them
# from the paths visible in your logs. Every line marked  # CHECK  is a guess:
# compare with your original setup cell. Names already defined are kept.
# ============================================================
import gc, json, math, random, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import yaml
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from ultralytics import YOLO
from ultralytics.nn.tasks import SegmentationModel


def _default(name, value):
    if name not in globals():
        globals()[name] = value


# ---- run configuration: 100 epochs (set EPOCHS below) ----
EPOCHS = 100                               # <<< the ONLY place to change the epoch count
SSL_EPOCHS = EPOCHS                        # Stage A: self-supervised pretraining
FT_EPOCHS = EPOCHS                         # Stage B: supervised fine-tuning
EPOCH_TICKS = (list(range(1, FT_EPOCHS + 1)) if FT_EPOCHS <= 20
               else [1] + list(range(10, FT_EPOCHS + 1, 10)))   # readable x-ticks for long runs
plt.rcParams["lines.markersize"] = 6 if FT_EPOCHS <= 20 else 3   # small markers for 100 points
_default("SEED", 42)                        # Step 1: seed = 42
_default("NC", 6)                           # six disease/health classes
_default("SC", "s")                         # YOLO26s
_default("IMG_SIZE", 640)                   # YOLO input size (Step 2)
_default("SSL_SIZE", 224)                   # CHECK: SSL crop size
_default("SSL_BATCH", 64)                   # CHECK
_default("TEACHER_BATCH", 64)               # CHECK
_default("FT_BATCH", 16)                    # CHECK
_default("WORKERS", 2)                      # CHECK
_default("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# ---- paths (taken from your logs: /kaggle/working/yolo_ds and .../eval/figures) ----
_default("W", Path("/kaggle/working"))
_default("DS", W / "yolo_ds")
# folder name follows the epoch count: .../papaya_real_dinov3_mae_e100/eval/figures
TAG = f"real_dinov3_mae_e{EPOCHS}"
EXP_ROOT = W / f"papaya_{TAG}"
_default("CKPT", EXP_ROOT / "checkpoints")                       # CHECK
_default("LOGS", EXP_ROOT / "logs")                              # CHECK
_default("RUNS", EXP_ROOT / "runs")                              # CHECK
_default("EV", EXP_ROOT / "eval")
_default("FIG", EV / "figures")
for _p in (DS, EXP_ROOT, CKPT, LOGS, RUNS, EV, FIG):
    _p.mkdir(parents=True, exist_ok=True)


def seed_all(seed=None):
    seed = SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_all()
print("SSL_EPOCHS =", SSL_EPOCHS, "| FT_EPOCHS =", FT_EPOCHS, "| SEED =", SEED)
print("Experiment folder:", EXP_ROOT)


In [ ]:
# ============================================================
# CELL 2 — LOCATE AND EXTRACT THE ANNOTATED DATASET
# Purpose: Find papaya_annotated or annotated.zip in Kaggle.
# ============================================================

INPUT = Path("/kaggle/input")

def find_dataset_root(search_root):
    for file in search_root.rglob("class_names.json"):
        root = file.parent

        if all(
            (root / name).exists()
            for name in ["images", "masks", "manifests"]
        ):
            return root

    return None

ROOT = find_dataset_root(INPUT)

if ROOT is None:
    archives = list(INPUT.rglob("annotated.zip"))

    if archives:
        extraction_dir = W / "dataset_extract"
        extraction_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.namelist():
                destination = (
                    extraction_dir / item
                ).resolve()

                assert destination.is_relative_to(
                    extraction_dir.resolve()
                )

            archive.extractall(extraction_dir)

        ROOT = find_dataset_root(extraction_dir)

# An already-prepared yolo_ds can be reused.
PREPARED = (
    (DS / "data.yaml").exists()
    and (DS / "split_table.csv").exists()
    and (DS / "class_map.json").exists()
)

assert ROOT is not None or PREPARED, (
    "Attach annotated.zip or the prepared dataset."
)

print("Original dataset:", ROOT)
print("Prepared YOLO dataset available:", PREPARED)

In [ ]:
# ============================================================
# CELL 3 — DATASET VALIDATION
# Purpose: Verify split sizes, leakage, masks, and polygons.
# ============================================================

SPLITS = ["train", "val", "test"]

if ROOT is not None:
    man = pd.concat([
        pd.read_csv(ROOT / "manifests" / f"{s}.csv")
        for s in SPLITS
    ], ignore_index=True)

    class_ids = json.loads(
        (ROOT / "class_names.json").read_text()
    )

    YOLO_NAMES = {
        int(v) - 1: k
        for k, v in class_ids.items()
        if int(v) > 0
    }

    MASK_NAMES = {0: "background"}
    MASK_NAMES.update({
        int(v): k
        for k, v in class_ids.items()
        if int(v) > 0
    })

else:
    man = pd.read_csv(
        DS / "split_table.csv"
    )

    class_map = json.loads(
        (DS / "class_map.json").read_text()
    )

    YOLO_NAMES = {
        int(k): v
        for k, v in class_map["yolo_id_to_name"].items()
    }

    MASK_NAMES = {
        int(k): v
        for k, v in class_map["mask_value_to_name"].items()
    }

print("Dataset size:", len(man))
print("\nSplit counts:")
print(man["split"].value_counts())

assert man["uid"].is_unique
assert len(man) == 1358
assert set(man["split"]) == set(SPLITS)
assert len(YOLO_NAMES) == NC

for key in ["group", "md5", "source_image"]:
    if key in man.columns:
        leakage = (
            man.groupby(key)["split"]
            .nunique().gt(1).sum()
        )

        print(f"{key} cross-split leakage:", leakage)
        assert leakage == 0

# Validate the annotation files where the original dataset exists.
if ROOT is not None:
    for row in tqdm(
        man.itertuples(index=False),
        total=len(man),
        desc="Checking masks and polygons"
    ):
        for col in ["image", "mask", "yolo_seg"]:
            assert (ROOT / getattr(row, col)).exists()

        mask = np.asarray(
            Image.open(ROOT / row.mask)
        )

        if mask.ndim == 3:
            mask = mask[..., 0]

        expected = int(row.label)

        assert set(np.unique(mask)).issubset(
            {0, expected}
        ), row.uid

        lines = (
            ROOT / row.yolo_seg
        ).read_text().splitlines()

        assert lines, f"Empty annotation: {row.uid}"

        for line in lines:
            fields = line.split()

            cls = int(fields[0])
            coords = np.asarray(
                fields[1:], dtype=np.float32
            )

            assert cls == expected - 1
            assert len(coords) >= 6
            assert len(coords) % 2 == 0
            assert np.all(
                (coords >= 0) & (coords <= 1)
            )

print("\nDataset validation completed.")

In [ ]:
# ============================================================
# CELL 4 — BUILD YOLO26 SEGMENTATION DATASET
# Purpose: Prepare images, YOLO polygons, and semantic masks.
# ============================================================

EXPECTED_COUNTS = {
    "train": 944,
    "val": 207,
    "test": 207
}

def dataset_complete():
    for split, expected in EXPECTED_COUNTS.items():
        for sub in ["images", "labels", "masks"]:
            folder = DS / sub / split

            if not folder.exists():
                return False

            if len(list(folder.iterdir())) != expected:
                return False

    return True

if dataset_complete():
    print("Existing YOLO dataset found.")
    print("Skipping unnecessary file copies.")

else:
    assert ROOT is not None, (
        "Original annotated dataset required for copying."
    )

    for split in SPLITS:
        for sub in ["images", "labels", "masks"]:
            (DS / sub / split).mkdir(
                parents=True, exist_ok=True
            )

    for row in tqdm(
        man.itertuples(index=False),
        total=len(man),
        desc="Preparing YOLO files"
    ):
        split = row.split
        uid = str(row.uid)

        extension = Path(row.image).suffix.lower()

        shutil.copy2(
            ROOT / row.image,
            DS / "images" / split / f"{uid}{extension}"
        )

        shutil.copy2(
            ROOT / row.yolo_seg,
            DS / "labels" / split / f"{uid}.txt"
        )

        shutil.copy2(
            ROOT / row.mask,
            DS / "masks" / split / f"{uid}.png"
        )

assert dataset_complete()

DATA_YAML = DS / "data.yaml"

with DATA_YAML.open("w") as f:
    yaml.safe_dump({
        "path": str(DS),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": YOLO_NAMES
    }, f, sort_keys=False)

(DS / "class_map.json").write_text(
    json.dumps({
        "yolo_id_to_name": YOLO_NAMES,
        "mask_value_to_name": MASK_NAMES
    }, indent=2)
)

man.to_csv(
    DS / "split_table.csv",
    index=False
)

SSL_FILES = sorted(
    str(p)
    for p in (DS / "images/train").iterdir()
    if p.is_file()
)

(DS / "ssl_train_images.txt").write_text(
    "\n".join(SSL_FILES)
)

for split in SPLITS:
    print(
        split,
        {
            sub: len(list((DS / sub / split).iterdir()))
            for sub in ["images", "labels", "masks"]
        }
    )

print("SSL training images:", len(SSL_FILES))
print("Dataset preparation complete.")

In [ ]:
# ============================================================
# CELL 5 — YOLO26s BACKBONE
# Purpose: Extract features and transfer SSL weights later.
# ============================================================

def new_full_model():
    return SegmentationModel(
        f"yolo26{SC}-seg.yaml",
        nc=NC,
        verbose=False
    )

# Determine backbone length from YOLO architecture.
probe_model = new_full_model()
N_BB = len(probe_model.yaml["backbone"])

class YoloBackbone(nn.Module):
    def __init__(self, full_model, out_idx=None):
        super().__init__()

        self.layers = nn.ModuleList(
            list(full_model.model)[:N_BB]
        )

        self.out_idx = (
            [N_BB - 1]
            if out_idx is None
            else list(out_idx)
        )

    def forward(self, x):
        outputs = []

        for layer in self.layers:
            if layer.f != -1:
                if isinstance(layer.f, int):
                    x = outputs[layer.f]
                else:
                    x = [
                        x if index == -1
                        else outputs[index]
                        for index in layer.f
                    ]

            x = layer(x)
            outputs.append(x)

        return [
            outputs[index]
            for index in self.out_idx
        ]


def backbone_state(full_model):
    return {
        k: v.detach().cpu().clone()
        for k, v in full_model.state_dict().items()
        if k.startswith("model.")
        and int(k.split(".")[1]) < N_BB
    }


def transfer_backbone(state, target):
    destination = target.state_dict()

    matched = {
        k: v
        for k, v in state.items()
        if k in destination
        and destination[k].shape == v.shape
    }

    assert len(matched) == len(state), (
        f"Transfer mismatch: {len(matched)}/{len(state)}"
    )

    target.load_state_dict(
        matched, strict=False
    )

    return len(matched)


# Detect feature channel dimensions instead of assuming them.
with torch.no_grad():
    backbone_probe = YoloBackbone(
        probe_model,
        out_idx=[4, 6, N_BB - 1]
    ).eval()

    features = backbone_probe(
        torch.zeros(1, 3, SSL_SIZE, SSL_SIZE)
    )

    FEATURE_CHANNELS = [
        f.shape[1] for f in features
    ]

print("Backbone layers:", N_BB)
print("Feature channels:", FEATURE_CHANNELS)
print("Feature resolutions:", [
    tuple(f.shape[-2:]) for f in features
])

del backbone_probe, probe_model, features
gc.collect()

In [ ]:
# ============================================================
# CELL 6 — SHARED SSL DATASET AND AUGMENTATION UTILITIES
# Purpose: Load training images without using class labels.
# ============================================================

SSL_FILES = [
    line.strip()
    for line in (
        DS / "ssl_train_images.txt"
    ).read_text().splitlines()
    if line.strip()
]

assert len(SSL_FILES) == 944

assert all(
    "/images/train/" in path
    for path in SSL_FILES
)


class SSLDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as im:
            image = im.convert("RGB")
            return self.transform(image)


def make_loader(transform):
    return DataLoader(
        SSLDataset(SSL_FILES, transform),
        batch_size=SSL_BATCH,
        shuffle=True,
        drop_last=True,
        num_workers=WORKERS,
        pin_memory=True
    )


def base_augmentation(size, scale):
    return T.Compose([
        T.RandomResizedCrop(
            size, scale=scale
        ),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomApply([
            T.ColorJitter(.4, .4, .2, .05)
        ], p=.8),
        T.RandomGrayscale(p=.1),
        T.ToTensor()
    ])


def make_mlp(input_dim, hidden_dim, output_dim):
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),
        nn.Linear(hidden_dim, output_dim)
    )


def epoch_lr(epoch):
    # Cosine decay across SSL_EPOCHS epochs.
    progress = epoch / max(SSL_EPOCHS - 1, 1)

    return 1e-5 + .5 * (
        .001 - 1e-5
    ) * (1 + math.cos(math.pi * progress))


def save_ssl_history(method, history):
    pd.DataFrame(history).to_csv(
        LOGS / f"{method}_yolo26s_{TAG}_history.csv",
        index=False
    )


def cleanup_gpu():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("SSL data loader ready.")
print("SSL epochs:", SSL_EPOCHS)
print("Learning rates:", [
    round(epoch_lr(e), 7)
    for e in range(SSL_EPOCHS)
])

In [ ]:
# ============================================================
# CELL 6A — HUGGING FACE AUTHENTICATION FOR DINOV3
# ============================================================

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
from transformers import AutoConfig

DINO_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"

# Securely retrieve the token from Kaggle Secrets.
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_Token")
except Exception as exc:
    raise RuntimeError(
        "HF_TOKEN is unavailable. Add it in Kaggle "
        "Add-ons > Secrets and enable notebook access."
    ) from exc

assert HF_TOKEN, "HF_TOKEN is empty."

# Verify that the token identifies a valid HF account.
account = HfApi(token=HF_TOKEN).whoami()
print("Authenticated Hugging Face account:", account["name"])

# Verify access to the actual gated model.
try:
    config = AutoConfig.from_pretrained(
        DINO_ID,
        token=HF_TOKEN
    )

    print("DINOv3 repository access verified.")
    print("Model type:", config.model_type)

except Exception as exc:
    raise RuntimeError(
        "DINOv3 access is not available to this account. "
        "Confirm that the account has accepted the model "
        "conditions and that its token has read permission."
    ) from exc

In [ ]:

# ============================================================
# CELL 7 — GENUINE PRETRAINED DINOV3 -> YOLO26 DISTILLATION
# Purpose: Transfer pretrained DINOv3 knowledge to YOLO26s.
# Training: Exactly 5 student distillation epochs.
# Teacher: Frozen pretrained DINOv3 ViT-S/16.
# ============================================================

from transformers import AutoModel
from kaggle_secrets import UserSecretsClient

# ------------------------------------------------------------
# STEP 1 — CONFIGURATION
# ------------------------------------------------------------

DINO_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"

DINO_FINAL = (
    CKPT / f"dino_yolo26s_{TAG}_backbone_final.pt"
)

assert SSL_EPOCHS == EPOCHS, "SSL_EPOCHS must equal EPOCHS."

# Retrieve the Hugging Face token securely.
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN is missing from Kaggle Secrets."


# ------------------------------------------------------------
# STEP 2 — DEFINE THE YOLO26 STUDENT NETWORK
# ------------------------------------------------------------

class GlobalDistillStudent(nn.Module):

    def __init__(
        self,
        full,
        student_dim=512,
        teacher_dim=384
    ):
        super().__init__()

        self.backbone = YoloBackbone(full)

        self.projector = nn.Sequential(
            nn.Linear(student_dim, 512),
            nn.GELU(),
            nn.Linear(512, teacher_dim)
        )

    def forward(self, x):

        features = self.backbone(x)[0]

        pooled = features.mean(dim=(2, 3))

        projected = self.projector(pooled)

        return F.normalize(
            projected.float(),
            dim=-1
        )


# ------------------------------------------------------------
# STEP 3 — DINOV3 INPUT NORMALIZATION
# ------------------------------------------------------------

DINOV3_MEAN = torch.tensor(
    [.485, .456, .406],
    device=DEVICE
).view(1, 3, 1, 1)

DINOV3_STD = torch.tensor(
    [.229, .224, .225],
    device=DEVICE
).view(1, 3, 1, 1)


# ------------------------------------------------------------
# STEP 4 — CHECK WHETHER TRAINING IS ALREADY COMPLETE
# ------------------------------------------------------------

if DINO_FINAL.exists():

    checkpoint = torch.load(
        DINO_FINAL,
        map_location="cpu",
        weights_only=False
    )

    assert (
        checkpoint.get("method")
        == "pretrained_dinov3_distillation"
    )

    assert checkpoint["epochs"] == SSL_EPOCHS

    print("Existing genuine DINOv3 checkpoint found.")
    print("Skipping completed training:", DINO_FINAL)

else:

    # --------------------------------------------------------
    # STEP 5 — INITIALIZE TRAINING
    # --------------------------------------------------------

    seed_all()

    print("Loading pretrained DINOv3 teacher...")

    dino_teacher = AutoModel.from_pretrained(
        DINO_ID,
        token=HF_TOKEN,
        attn_implementation="sdpa"
    ).to(DEVICE).eval()

    # Freeze teacher parameters.
    dino_teacher.requires_grad_(False)

    teacher_dim = dino_teacher.config.hidden_size

    print("DINOv3 teacher loaded successfully.")
    print("Teacher hidden dimension:", teacher_dim)

    # Initialize YOLO26 student.
    student_full = new_full_model()

    student = GlobalDistillStudent(
        student_full,
        student_dim=FEATURE_CHANNELS[-1],
        teacher_dim=teacher_dim
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=3e-4,
        weight_decay=1e-4
    )

    # --------------------------------------------------------
    # STEP 6 — CREATE DATA LOADER
    # --------------------------------------------------------

    transform = T.Compose([
        T.RandomResizedCrop(
            224,
            scale=(.6, 1.0)
        ),
        T.RandomHorizontalFlip(),
        T.ToTensor()
    ])

    loader = DataLoader(
        SSLDataset(SSL_FILES, transform),
        batch_size=TEACHER_BATCH,
        shuffle=True,
        num_workers=WORKERS,
        pin_memory=True
    )

    # --------------------------------------------------------
    # STEP 7 — TRAIN STUDENT FOR 5 EPOCHS
    # --------------------------------------------------------

    history = []

    for epoch in range(SSL_EPOCHS):

        student.train()

        losses = []

        progress = tqdm(
            loader,
            desc=(
                f"DINOv3 Distillation "
                f"{epoch+1}/{SSL_EPOCHS}"
            )
        )

        for images in progress:

            images = images.to(DEVICE)

            # -----------------------------------------------
            # TEACHER FEATURE EXTRACTION
            # -----------------------------------------------

            with torch.no_grad():

                normalized_images = (
                    images - DINOV3_MEAN
                ) / DINOV3_STD

                teacher_output = dino_teacher(
                    pixel_values=normalized_images
                )

                # Global image representation.
                if (
                    hasattr(teacher_output, "pooler_output")
                    and teacher_output.pooler_output is not None
                ):
                    target = teacher_output.pooler_output
                else:
                    target = (
                        teacher_output.last_hidden_state[:, 0, :]
                    )

                target = F.normalize(
                    target.float(),
                    dim=-1
                )

            # -----------------------------------------------
            # STUDENT FEATURE EXTRACTION
            # -----------------------------------------------

            prediction = student(images)

            # Cosine representation distillation.
            loss = 1 - (
                prediction * target
            ).sum(dim=-1).mean()

            # -----------------------------------------------
            # BACKPROPAGATION
            # -----------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()

            nn.utils.clip_grad_norm_(
                student.parameters(),
                max_norm=2.0
            )

            optimizer.step()

            losses.append(
                float(loss.item())
            )

            progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        # ----------------------------------------------------
        # STEP 8 — SAVE EPOCH RESULTS
        # ----------------------------------------------------

        average_loss = float(
            np.mean(losses)
        )

        history.append({
            "epoch": epoch + 1,
            "loss": average_loss,
            "objective": "dinov3_teacher_cosine_distillation"
        })

        save_ssl_history(
            "dino",
            history
        )

        print(
            f"DINOv3 Epoch {epoch+1}/{SSL_EPOCHS} "
            f"| Loss: {average_loss:.4f}"
        )

    # --------------------------------------------------------
    # STEP 9 — SAVE THE TRAINED YOLO26 BACKBONE
    # --------------------------------------------------------

    torch.save(
        {
            "backbone": backbone_state(student_full),
            "method": "pretrained_dinov3_distillation",
            "teacher_id": DINO_ID,
            "epochs": SSL_EPOCHS
        },
        DINO_FINAL
    )

    print("DINOv3 distillation completed.")

    # --------------------------------------------------------
    # STEP 10 — RELEASE GPU MEMORY
    # --------------------------------------------------------

    del dino_teacher
    del student
    del student_full
    del optimizer
    del loader

    cleanup_gpu()


# ------------------------------------------------------------
# FINAL OUTPUT
# ------------------------------------------------------------

print("DINOv3 teacher-guided YOLO26 backbone:")
print(DINO_FINAL)


In [ ]:
# ============================================================
# CELL 8 — BYOL SELF-SUPERVISED PRETRAINING
# Purpose: Train online and target networks for SSL_EPOCHS epochs.
# ============================================================

class BYOLViews:
    def __init__(self):
        self.augment = base_augmentation(
            SSL_SIZE, (.4, 1.0)
        )

    def __call__(self, image):
        return [
            self.augment(image),
            self.augment(image)
        ]


class BYOLNetwork(nn.Module):
    def __init__(self, full, predictor=True):
        super().__init__()

        self.backbone = YoloBackbone(full)

        self.projector = make_mlp(
            FEATURE_CHANNELS[-1], 4096, 256
        )

        self.predictor = (
            make_mlp(256, 4096, 256)
            if predictor else None
        )

    def forward(self, x):
        feature = self.backbone(x)[0]
        pooled = feature.mean(dim=(2, 3))

        projection = self.projector(pooled)

        prediction = (
            self.predictor(projection)
            if self.predictor is not None
            else None
        )

        return projection, prediction


def byol_loss(prediction, target):
    return 2 - 2 * F.cosine_similarity(
        prediction.float(),
        target.float().detach(),
        dim=-1
    ).mean()


BYOL_FINAL = (
    CKPT / f"byol_yolo26s_{TAG}_backbone_final.pt"
)

BYOL_LAST = (
    CKPT / f"byol_yolo26s_{TAG}_last.pt"
)

if BYOL_FINAL.exists():
    print(f"Existing {SSL_EPOCHS}-epoch BYOL checkpoint found.")

else:
    seed_all()

    online_full = new_full_model()
    target_full = new_full_model()

    target_full.load_state_dict(
        online_full.state_dict()
    )

    online = BYOLNetwork(
        online_full, predictor=True
    ).to(DEVICE)

    target = BYOLNetwork(
        target_full, predictor=False
    ).to(DEVICE)

    target.backbone.load_state_dict(
        online.backbone.state_dict()
    )

    target.projector.load_state_dict(
        online.projector.state_dict()
    )

    for p in target.parameters():
        p.requires_grad = False

    optimizer = torch.optim.AdamW(
        online.parameters(),
        lr=.001,
        weight_decay=.04
    )

    scaler = torch.amp.GradScaler("cuda")
    loader = make_loader(BYOLViews())

    history = []
    start_epoch = 0

    if BYOL_LAST.exists():
        ck = torch.load(
            BYOL_LAST,
            map_location=DEVICE,
            weights_only=False
        )

        online.load_state_dict(ck["online"])
        target.load_state_dict(ck["target"])
        optimizer.load_state_dict(ck["optimizer"])
        scaler.load_state_dict(ck["scaler"])

        history = ck["history"]
        start_epoch = ck["epoch"]

    for epoch in range(start_epoch, SSL_EPOCHS):
        online.train()
        target.train()

        lr = epoch_lr(epoch)

        for group in optimizer.param_groups:
            group["lr"] = lr

        losses = []

        for views in tqdm(
            loader,
            desc=f"BYOL epoch {epoch+1}/{SSL_EPOCHS}"
        ):
            v1, v2 = [
                x.to(DEVICE)
                for x in views
            ]

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                "cuda", dtype=torch.float16
            ):
                _, p1 = online(v1)
                _, p2 = online(v2)

                with torch.no_grad():
                    t1, _ = target(v1)
                    t2, _ = target(v2)

                loss = (
                    byol_loss(p1, t2)
                    + byol_loss(p2, t1)
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            nn.utils.clip_grad_norm_(
                online.parameters(), 3.0
            )

            scaler.step(optimizer)
            scaler.update()

            momentum = .996 + (
                1 - .996
            ) * (epoch / SSL_EPOCHS)

            with torch.no_grad():
                for online_module, target_module in [
                    (online.backbone, target.backbone),
                    (online.projector, target.projector)
                ]:
                    for op, tp in zip(
                        online_module.parameters(),
                        target_module.parameters()
                    ):
                        tp.mul_(momentum).add_(
                            op, alpha=1 - momentum
                        )

        # Important: compute the epoch average inside the batch loop.
            losses.append(loss.item())

        avg_loss = float(np.mean(losses))

        history.append({
            "epoch": epoch + 1,
            "loss": avg_loss,
            "lr": lr
        })

        save_ssl_history("byol", history)

        torch.save({
            "epoch": epoch + 1,
            "online": online.state_dict(),
            "target": target.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "history": history
        }, BYOL_LAST)

        print(
            f"BYOL epoch {epoch+1}/{SSL_EPOCHS} | "
            f"Loss: {avg_loss:.4f}"
        )

    torch.save({
        "backbone": backbone_state(target_full),
        "method": "byol",
        "epochs": SSL_EPOCHS
    }, BYOL_FINAL)

    del online, target
    del online_full, target_full
    del optimizer, loader, scaler

    cleanup_gpu()

print("BYOL SSL stage completed.")

In [ ]:
# ============================================================
# CELL 9 — STANDARD ViT-MAE RECONSTRUCTION + YOLO DISTILLATION
# Standard MAE uses visible patch tokens and Transformer decoder.
# Step A: SSL_EPOCHS epochs MAE reconstruction on training images.
# Step B: SSL_EPOCHS epochs distill the resulting MAE encoder into YOLO.
# ============================================================
from transformers import ViTMAEForPreTraining

MAE_ID = 'facebook/vit-mae-base'
MAE_FINAL = CKPT / f'mae_yolo26s_{TAG}_backbone_final.pt'
MAE_TEACHER = CKPT / f'mae_vit_teacher_{TAG}.pt'

mae_loader = DataLoader(SSLDataset(SSL_FILES, T.Compose([
    T.RandomResizedCrop(224, scale=(.6,1.)), T.RandomHorizontalFlip(), T.ToTensor()
])), batch_size=TEACHER_BATCH, shuffle=True, num_workers=WORKERS, pin_memory=True)

if MAE_FINAL.exists():
    ck=torch.load(MAE_FINAL, map_location='cpu', weights_only=False)
    assert ck.get('method') == 'standard_vit_mae_then_yolo_distillation' and ck['epochs']==SSL_EPOCHS
    print('Real MAE-derived YOLO backbone already available:', MAE_FINAL)
else:
    seed_all()
    # This is genuine pretrained MAE, not a YOLO masked-input imitation.
    mae_teacher = ViTMAEForPreTraining.from_pretrained(MAE_ID).to(DEVICE)
    mean=torch.tensor([.485,.456,.406],device=DEVICE).view(1,3,1,1)
    std=torch.tensor([.229,.224,.225],device=DEVICE).view(1,3,1,1)

    if MAE_TEACHER.exists():
        mae_teacher.load_state_dict(torch.load(MAE_TEACHER,map_location='cpu',weights_only=True))
        print('Restored completed MAE teacher fine-tuning.')
    else:
        opt_mae=torch.optim.AdamW(mae_teacher.parameters(),lr=1e-5,weight_decay=.05)
        reconstruction_history=[]
        for epoch in range(SSL_EPOCHS):
            mae_teacher.train(); losses=[]
            for images in tqdm(mae_loader,desc=f'Standard MAE reconstruction {epoch+1}/{SSL_EPOCHS}'):
                images=images.to(DEVICE)
                out=mae_teacher(pixel_values=(images-mean)/std)
                loss=out.loss
                opt_mae.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(mae_teacher.parameters(),1)
                opt_mae.step()
                losses.append(float(loss.item()))
            reconstruction_history.append({'epoch':epoch+1,'loss':float(np.mean(losses)), 'objective':'standard_mae_masked_patch_mse'})
            pd.DataFrame(reconstruction_history).to_csv(LOGS/f'mae_reconstruction_{TAG}_history.csv',index=False)
            print('MAE reconstruction epoch',epoch+1,'loss',reconstruction_history[-1]['loss'])
        torch.save({k:v.cpu() for k,v in mae_teacher.state_dict().items()}, MAE_TEACHER)
        del opt_mae
        cleanup_gpu()

    # Distillation uses MAE encoder's CLS representation on visible tokens.
    # We deliberately use mask_ratio=0 during this feature-transfer stage
    # so both student and teacher see the same full image.
    mae_teacher.eval().requires_grad_(False)
    old_mask_ratio=mae_teacher.config.mask_ratio
    mae_teacher.config.mask_ratio=0.0
    student_full=new_full_model()
    student=GlobalDistillStudent(student_full, FEATURE_CHANNELS[-1],mae_teacher.config.hidden_size).to(DEVICE)
    optimizer=torch.optim.AdamW(student.parameters(),lr=3e-4,weight_decay=1e-4)
    history=[]
    for epoch in range(SSL_EPOCHS):
        student.train(); losses=[]
        for images in tqdm(mae_loader,desc=f'MAE -> YOLO distillation {epoch+1}/{SSL_EPOCHS}'):
            images=images.to(DEVICE)
            with torch.no_grad():
                teacher_features=mae_teacher.vit(pixel_values=(images-mean)/std).last_hidden_state[:,0]
                target=F.normalize(teacher_features.float(),dim=-1)
            pred=student(images)
            loss=1-(pred*target).sum(-1).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(student.parameters(),2)
            optimizer.step()
            losses.append(float(loss.item()))
        history.append({'epoch':epoch+1,'loss':float(np.mean(losses)),'objective':'mae_encoder_cosine_distillation'})
        save_ssl_history('mae',history)
        print('MAE transfer epoch',epoch+1,'loss',history[-1]['loss'])
    mae_teacher.config.mask_ratio=old_mask_ratio
    torch.save({'backbone':backbone_state(student_full),'method':'standard_vit_mae_then_yolo_distillation',
                'teacher_id':MAE_ID,'epochs':SSL_EPOCHS, 'mae_reconstruction_epochs':SSL_EPOCHS},MAE_FINAL)
    del mae_teacher, student,student_full,optimizer
    cleanup_gpu()
print('Standard MAE-derived YOLO backbone:',MAE_FINAL)


In [ ]:
# SSL pretraining loss — DINOv3 only (not directly comparable across objectives)
path = LOGS / f"dino_yolo26s_{TAG}_history.csv"
h = pd.read_csv(path)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h["epoch"], h["loss"], marker="o", linewidth=2)
ax.set(xlabel="SSL epoch", ylabel="Training objective")
ax.set_title("YOLOv26s (DINOv3) - SSL Pretraining Loss", fontsize=13, fontweight="bold")
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(FIG / "dino_ssl_loss_modelwise.png", dpi=150)
plt.show()

In [ ]:
# SSL pretraining loss — BYOL only (not directly comparable across objectives)
path = LOGS / f"byol_yolo26s_{TAG}_history.csv"
h = pd.read_csv(path)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h["epoch"], h["loss"], marker="o", linewidth=2)
ax.set(xlabel="SSL epoch", ylabel="Training objective")
ax.set_title("YOLOv26s (BYOL) - SSL Pretraining Loss", fontsize=13, fontweight="bold")
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(FIG / "byol_ssl_loss_modelwise.png", dpi=150)
plt.show()

In [ ]:
# SSL pretraining loss — MAE only (not directly comparable across objectives)
path = LOGS / f"mae_yolo26s_{TAG}_history.csv"
h = pd.read_csv(path)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h["epoch"], h["loss"], marker="o", linewidth=2)
ax.set(xlabel="SSL epoch", ylabel="Training objective")
ax.set_title("YOLOv26s (MAE) - SSL Pretraining Loss", fontsize=13, fontweight="bold")
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig(FIG / "mae_ssl_loss_modelwise.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# SHARED FINE-TUNING UTILITIES — NO TRAINING IN THIS CELL
# Purpose: Train each model for exactly FT_EPOCHS epochs.
# ============================================================

EXPS = ["baseline", "dino", "byol", "mae"]

SSL_SOURCES = {
    "baseline": None,
    "dino": DINO_FINAL,
    "byol": BYOL_FINAL,
    "mae": MAE_FINAL
}

FT_CONFIG = {
    "epochs": FT_EPOCHS,       # = FT_EPOCHS
    "imgsz": IMG_SIZE,
    "batch": FT_BATCH,
    "optimizer": "AdamW",
    "lr0": .001,
    "lrf": .01,
    "warmup_epochs": 1.0,
    "weight_decay": .0005,
    "cos_lr": False,
    "close_mosaic": 0,
    "patience": 0,             # No early stopping
    "seed": SEED,
    "deterministic": True,
    "workers": WORKERS,
    "amp": True,
    "device": 0,
    "plots": False,   # avoid automatic baseline plots; custom model-wise plots follow
    "pretrained": False,
    "save": True
}

assert FT_CONFIG["epochs"] == FT_EPOCHS

(EXP_ROOT / "ft_config.json").write_text(
    json.dumps({
        "epochs": FT_EPOCHS,
        "settings": FT_CONFIG,
        "sources": {
            k: str(v) if v else None
            for k, v in SSL_SOURCES.items()
        }
    }, indent=2)
)


def train_experiment(method):
    run_name = f"{method}_real_ssl_e{FT_EPOCHS}"
    run_dir = RUNS / run_name

    best = run_dir / "weights/best.pt"
    last = run_dir / "weights/last.pt"
    done = run_dir / "DONE.json"

    if done.exists() and best.exists():
        print("Already completed in NEW genuine SSL experiment:", method)
        return str(best)

    source_state = None

    if SSL_SOURCES[method] is not None:
        checkpoint = torch.load(
            SSL_SOURCES[method],
            map_location="cpu",
            weights_only=False
        )

        assert checkpoint["epochs"] == SSL_EPOCHS
        source_state = checkpoint["backbone"]

    if last.exists():
        # Resume an interrupted run.
        print("Resuming:", method)

        model = YOLO(str(last))
        model.train(resume=True, device=0)

    else:
        # Start a fresh supervised training run.
        assert not run_dir.exists(), (
            f"Existing incomplete run: {run_dir}"
        )

        seed_all(SEED)

        model = YOLO(
            f"yolo26{SC}-seg.yaml"
        )

        init_info = {
            "method": method,
            "ssl_epochs": SSL_EPOCHS,
            "finetune_epochs": FT_EPOCHS
        }

        def inject_backbone(trainer):
            if source_state is not None:
                transferred = transfer_backbone(
                    source_state,
                    trainer.model
                )

                init_info["transferred"] = transferred

                if getattr(trainer, "ema", None) is not None:
                    transfer_backbone(
                        source_state,
                        trainer.ema.ema
                    )

        def verify_initialization(trainer):
            if source_state is None:
                init_info["initialization"] = "random"
                print("Baseline: random initialization")

            else:
                current = trainer.model.state_dict()

                identical = all(
                    torch.equal(
                        current[k].detach().cpu(),
                        v
                    )
                    for k, v in source_state.items()
                )

                init_info["weights_identical"] = identical

                assert identical, (
                    f"SSL transfer failed: {method}"
                )

                print(
                    method,
                    "SSL weights transferred:",
                    init_info.get("transferred")
                )

            Path(trainer.save_dir).mkdir(
                parents=True, exist_ok=True
            )

            (
                Path(trainer.save_dir)
                / "init_check.json"
            ).write_text(
                json.dumps(init_info, indent=2)
            )

        model.add_callback(
            "on_pretrain_routine_end",
            inject_backbone
        )

        model.add_callback(
            "on_train_start",
            verify_initialization
        )

        model.train(
            data=str(DATA_YAML),
            project=str(RUNS),
            name=run_name,
            exist_ok=False,
            **FT_CONFIG
        )

    assert best.exists(), (
        f"Missing best.pt for {method}"
    )

    done.write_text(
        json.dumps({
            "method": method,
            "epochs": FT_EPOCHS,
            "checkpoint": str(best)
        }, indent=2)
    )

    del model
    cleanup_gpu()

    return str(best)

In [ ]:
# MODEL TRAINING — YOLO26s (one model, one cell)
# Resume-safe: previously completed checkpoints are reused.
if "BEST_WEIGHTS" not in globals():
    BEST_WEIGHTS = {}
print("Training YOLO26s, epochs:", FT_EPOCHS)
BEST_WEIGHTS["baseline"] = train_experiment("baseline")
print("Saved YOLO26s checkpoint:", BEST_WEIGHTS["baseline"])

In [ ]:
# MODEL TRAINING — YOLO26s + DINOv3 (one model, one cell)
# Resume-safe: previously completed checkpoints are reused.
if "BEST_WEIGHTS" not in globals():
    BEST_WEIGHTS = {}
print("Training YOLO26s + DINOv3, epochs:", FT_EPOCHS)
BEST_WEIGHTS["dino"] = train_experiment("dino")
print("Saved YOLO26s + DINOv3 checkpoint:", BEST_WEIGHTS["dino"])

In [ ]:
# MODEL TRAINING — YOLO26s + BYOL (one model, one cell)
# Resume-safe: previously completed checkpoints are reused.
if "BEST_WEIGHTS" not in globals():
    BEST_WEIGHTS = {}
print("Training YOLO26s + BYOL, epochs:", FT_EPOCHS)
BEST_WEIGHTS["byol"] = train_experiment("byol")
print("Saved YOLO26s + BYOL checkpoint:", BEST_WEIGHTS["byol"])

In [ ]:
# MODEL TRAINING — YOLO26s + MAE (one model, one cell)
# Resume-safe: previously completed checkpoints are reused.
if "BEST_WEIGHTS" not in globals():
    BEST_WEIGHTS = {}
print("Training YOLO26s + MAE, epochs:", FT_EPOCHS)
BEST_WEIGHTS["mae"] = train_experiment("mae")
print("Saved YOLO26s + MAE checkpoint:", BEST_WEIGHTS["mae"])

In [ ]:
# ============================================================
# CELL 11 — FINE-TUNE FOUR YOLO26s SEGMENTATION MODELS
# Purpose: Train each model for exactly FT_EPOCHS epochs.
# ============================================================

EXPS = ["baseline", "dino", "byol", "mae"]

SSL_SOURCES = {
    "baseline": None,
    "dino": DINO_FINAL,
    "byol": BYOL_FINAL,
    "mae": MAE_FINAL
}

FT_CONFIG = {
    "epochs": FT_EPOCHS,       # = FT_EPOCHS
    "imgsz": IMG_SIZE,
    "batch": FT_BATCH,
    "optimizer": "AdamW",
    "lr0": .001,
    "lrf": .01,
    "warmup_epochs": 1.0,
    "weight_decay": .0005,
    "cos_lr": False,
    "close_mosaic": 0,
    "patience": 0,             # No early stopping
    "seed": SEED,
    "deterministic": True,
    "workers": WORKERS,
    "amp": True,
    "device": 0,
    "plots": True,
    "pretrained": False,
    "save": True
}

assert FT_CONFIG["epochs"] == FT_EPOCHS

(EXP_ROOT / "ft_config.json").write_text(
    json.dumps({
        "epochs": FT_EPOCHS,
        "settings": FT_CONFIG,
        "sources": {
            k: str(v) if v else None
            for k, v in SSL_SOURCES.items()
        }
    }, indent=2)
)


def train_experiment(method):
    run_name = f"{method}_real_ssl_e{FT_EPOCHS}"
    run_dir = RUNS / run_name

    best = run_dir / "weights/best.pt"
    last = run_dir / "weights/last.pt"
    done = run_dir / "DONE.json"

    if done.exists() and best.exists():
        print("Already completed in NEW genuine SSL experiment:", method)
        return str(best)

    source_state = None

    if SSL_SOURCES[method] is not None:
        checkpoint = torch.load(
            SSL_SOURCES[method],
            map_location="cpu",
            weights_only=False
        )

        assert checkpoint["epochs"] == SSL_EPOCHS
        source_state = checkpoint["backbone"]

    if last.exists():
        # Resume an interrupted run.
        print("Resuming:", method)

        model = YOLO(str(last))
        model.train(resume=True, device=0)

    else:
        # Start a fresh supervised training run.
        assert not run_dir.exists(), (
            f"Existing incomplete run: {run_dir}"
        )

        seed_all(SEED)

        model = YOLO(
            f"yolo26{SC}-seg.yaml"
        )

        init_info = {
            "method": method,
            "ssl_epochs": SSL_EPOCHS,
            "finetune_epochs": FT_EPOCHS
        }

        def inject_backbone(trainer):
            if source_state is not None:
                transferred = transfer_backbone(
                    source_state,
                    trainer.model
                )

                init_info["transferred"] = transferred

                if getattr(trainer, "ema", None) is not None:
                    transfer_backbone(
                        source_state,
                        trainer.ema.ema
                    )

        def verify_initialization(trainer):
            if source_state is None:
                init_info["initialization"] = "random"
                print("Baseline: random initialization")

            else:
                current = trainer.model.state_dict()

                identical = all(
                    torch.equal(
                        current[k].detach().cpu(),
                        v
                    )
                    for k, v in source_state.items()
                )

                init_info["weights_identical"] = identical

                assert identical, (
                    f"SSL transfer failed: {method}"
                )

                print(
                    method,
                    "SSL weights transferred:",
                    init_info.get("transferred")
                )

            Path(trainer.save_dir).mkdir(
                parents=True, exist_ok=True
            )

            (
                Path(trainer.save_dir)
                / "init_check.json"
            ).write_text(
                json.dumps(init_info, indent=2)
            )

        model.add_callback(
            "on_pretrain_routine_end",
            inject_backbone
        )

        model.add_callback(
            "on_train_start",
            verify_initialization
        )

        model.train(
            data=str(DATA_YAML),
            project=str(RUNS),
            name=run_name,
            exist_ok=False,
            **FT_CONFIG
        )

    assert best.exists(), (
        f"Missing best.pt for {method}"
    )

    done.write_text(
        json.dumps({
            "method": method,
            "epochs": FT_EPOCHS,
            "checkpoint": str(best)
        }, indent=2)
    )

    del model
    cleanup_gpu()

    return str(best)


BEST_WEIGHTS = {}

for method in EXPS:
    print("\n" + "=" * 55)
    print("TRAINING:", method.upper())
    print("EPOCHS:", FT_EPOCHS)
    print("=" * 55)

    BEST_WEIGHTS[method] = train_experiment(
        method
    )

print(f"\nAll four {FT_EPOCHS}-epoch fine-tuning runs completed.")

In [ ]:
# ============================================================
# CELL 12 — CHECK TRAINING COMPLETION
# Purpose: Verify all four models finished FT_EPOCHS epochs.
# ============================================================

status_rows = []
BEST_WEIGHTS = {}

for method in EXPS:
    run_dir = RUNS / f"{method}_real_ssl_e{FT_EPOCHS}"

    best = run_dir / "weights/best.pt"
    last = run_dir / "weights/last.pt"
    results_file = run_dir / "results.csv"
    done = run_dir / "DONE.json"

    epochs_recorded = 0

    if results_file.exists():
        history = pd.read_csv(results_file)
        history.columns = [
            x.strip() for x in history.columns
        ]

        epochs_recorded = history[
            "epoch"
        ].nunique()

    status_rows.append({
        "model": method,
        "completed": done.exists(),
        "best_exists": best.exists(),
        "last_exists": last.exists(),
        "epochs_recorded": epochs_recorded
    })

    BEST_WEIGHTS[method] = str(best)

status_df = pd.DataFrame(status_rows)

display(status_df)

assert status_df["completed"].all()
assert status_df["best_exists"].all()
assert (
    status_df["epochs_recorded"] == FT_EPOCHS
).all()

print(f"All four models completed {FT_EPOCHS} epochs.")

In [ ]:
# ============================================================
# CELL 13 — SEMANTIC SEGMENTATION EVALUATION UTILITIES
# Purpose: Calculate IoU, Dice, precision, recall and F1.
# ============================================================

NUM_CLASSES = NC + 1  # Six classes + background

PRED_CONFIG = {
    "imgsz": IMG_SIZE,
    "conf": .25,
    "iou": .7,
    "max_det": 300,
    "retina_masks": True,
    "device": 0,
    "verbose": False
}


def semantic_from_result(result, shape):
    h, w = shape

    semantic = np.zeros(
        (h, w),
        dtype=np.uint8
    )

    if result.masks is None:
        return semantic

    masks = result.masks.data.cpu().numpy()

    classes = (
        result.boxes.cls.cpu()
        .numpy().astype(int)
    )

    confidence = (
        result.boxes.conf.cpu().numpy()
    )

    for index in np.argsort(confidence):
        mask = masks[index]

        if mask.shape != (h, w):
            mask = cv2.resize(
                mask,
                (w, h),
                interpolation=cv2.INTER_NEAREST
            )

        semantic[mask > .5] = (
            classes[index] + 1
        )

    return semantic


def confusion_matrix(gt, pred):
    indices = (
        gt.astype(np.int64) * NUM_CLASSES
        + pred.astype(np.int64)
    )

    return np.bincount(
        indices.ravel(),
        minlength=NUM_CLASSES ** 2
    ).reshape(NUM_CLASSES, NUM_CLASSES)


def metrics_from_cm(cm):
    cm = cm.astype(float)

    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    def safe_divide(numerator, denominator):
        return np.divide(
            numerator,
            denominator,
            out=np.full(
                numerator.shape, np.nan
            ),
            where=denominator > 0
        )

    iou = safe_divide(
        tp, tp + fp + fn
    )

    dice = safe_divide(
        2 * tp, 2 * tp + fp + fn
    )

    precision = safe_divide(
        tp, tp + fp
    )

    recall = safe_divide(
        tp, tp + fn
    )

    f1 = safe_divide(
        2 * tp, 2 * tp + fp + fn
    )

    summary = {
        "mIoU_7": float(np.nanmean(iou)),
        "mIoU_fg6": float(np.nanmean(iou[1:])),
        "mDice_7": float(np.nanmean(dice)),
        "mDice_fg6": float(np.nanmean(dice[1:])),
        "mPrecision_fg6": float(
            np.nanmean(precision[1:])
        ),
        "mRecall_fg6": float(
            np.nanmean(recall[1:])
        ),
        "mF1_fg6": float(
            np.nanmean(f1[1:])
        ),
        "pixel_accuracy": float(
            tp.sum() / max(cm.sum(), 1)
        )
    }

    per_class = pd.DataFrame({
        "IoU": iou,
        "Dice": dice,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }, index=[
        MASK_NAMES[i]
        for i in range(NUM_CLASSES)
    ])

    return summary, per_class


def evaluate_model(method, split):
    model = YOLO(BEST_WEIGHTS[method])

    paths = sorted(
        p for p in (
            DS / "images" / split
        ).iterdir()
        if p.is_file()
    )

    pred_dir = (
        EV / "predictions" / split / method
    )

    pred_dir.mkdir(
        parents=True, exist_ok=True
    )

    matrices = []
    uids = []
    no_prediction = 0

    for start in tqdm(
        range(0, len(paths), 8),
        desc=f"{method} {split}"
    ):
        batch = paths[start:start + 8]

        results = model.predict(
            [str(p) for p in batch],
            batch=len(batch),
            **PRED_CONFIG
        )

        for image_path, result in zip(
            batch, results
        ):
            uid = image_path.stem

            gt = np.asarray(
                Image.open(
                    DS / "masks" / split
                    / f"{uid}.png"
                )
            )

            if gt.ndim == 3:
                gt = gt[..., 0]

            pred = semantic_from_result(
                result, gt.shape
            )

            matrices.append(
                confusion_matrix(gt, pred)
            )

            uids.append(uid)

            if result.masks is None:
                no_prediction += 1

            Image.fromarray(pred).save(
                pred_dir / f"{uid}.png"
            )

    matrices = np.asarray(
        matrices, dtype=np.int64
    )

    np.savez_compressed(
        EV / f"{split}_{method}.npz",
        cms=matrices,
        uids=np.asarray(uids)
    )

    summary, per_class = metrics_from_cm(
        matrices.sum(axis=0)
    )

    summary["no_predictions"] = no_prediction

    del model
    cleanup_gpu()

    return summary, per_class


print("Semantic evaluation functions initialized.")

In [ ]:
# ============================================================
# CELL 14 — VALIDATION SET EVALUATION
# Purpose: Compare models and select using validation mIoU.
# ============================================================

validation_rows = {}
validation_classes = {}

for method in EXPS:
    metrics, per_class = evaluate_model(
        method, "val"
    )

    validation_rows[method] = metrics
    validation_classes[method] = per_class

val_summary = pd.DataFrame.from_dict(
    validation_rows,
    orient="index"
)

val_summary.to_csv(
    EV / "val_summary.csv"
)

for metric in [
    "IoU", "Dice", "Precision", "Recall", "F1"
]:
    table = pd.DataFrame({
        method: validation_classes[method][metric]
        for method in EXPS
    })

    table.to_csv(
        EV / f"val_per_class_{metric.lower()}.csv"
    )

SELECTED_MODEL = (
    val_summary["mIoU_7"].idxmax()
)

selection_path = EV / "selected_model.json"

if selection_path.exists():
    SELECTED_MODEL = json.loads(
        selection_path.read_text()
    )["selected"]

else:
    selection_path.write_text(
        json.dumps({
            "selected": SELECTED_MODEL,
            "criterion": "validation mIoU_7",
            "epochs": FT_EPOCHS
        }, indent=2)
    )

display(val_summary.round(4))

print("Validation-selected model:", SELECTED_MODEL)

In [ ]:
# ============================================================
# CELL 15 — FINAL TEST EVALUATION
# Purpose: Evaluate all models without changing selection.
# ============================================================

assert (
    EV / "selected_model.json"
).exists(), "Run validation selection first."

test_rows = {}
test_classes = {}

for method in EXPS:
    metrics, per_class = evaluate_model(
        method, "test"
    )

    test_rows[method] = metrics
    test_classes[method] = per_class

test_summary = pd.DataFrame.from_dict(
    test_rows,
    orient="index"
)

test_summary.to_csv(
    EV / "test_summary.csv"
)

for metric in [
    "IoU", "Dice", "Precision", "Recall", "F1"
]:
    table = pd.DataFrame({
        method: test_classes[method][metric]
        for method in EXPS
    })

    table.to_csv(
        EV / f"test_per_class_{metric.lower()}.csv"
    )

# Paired image-level bootstrap.
data = {
    method: np.load(
        EV / f"test_{method}.npz"
    )
    for method in EXPS
}

baseline_uids = data["baseline"]["uids"]
n_images = len(baseline_uids)

assert all(
    np.array_equal(
        data[method]["uids"], baseline_uids
    )
    for method in EXPS
)

rng = np.random.default_rng(SEED)

bootstrap_indices = rng.integers(
    0, n_images,
    size=(500, n_images)
)

bootstrap_scores = {}

for method in EXPS:
    cms = data[method]["cms"]
    scores = []

    for indices in bootstrap_indices:
        cm = cms[indices].sum(axis=0)

        metrics, _ = metrics_from_cm(cm)
        scores.append(metrics["mIoU_7"])

    bootstrap_scores[method] = np.array(
        scores
    )

bootstrap_rows = []

for method in EXPS:
    scores = bootstrap_scores[method]

    low, high = np.percentile(
        scores, [2.5, 97.5]
    )

    differences = (
        scores - bootstrap_scores["baseline"]
    )

    difference_low, difference_high = (
        np.percentile(
            differences, [2.5, 97.5]
        )
    )

    bootstrap_rows.append({
        "model": method,
        "CI_low": low,
        "CI_high": high,
        "difference_vs_baseline": (
            test_summary.loc[method, "mIoU_7"]
            - test_summary.loc["baseline", "mIoU_7"]
        ),
        "difference_CI_low": difference_low,
        "difference_CI_high": difference_high
    })

bootstrap_df = pd.DataFrame(
    bootstrap_rows
).set_index("model")

bootstrap_df.to_csv(
    EV / "test_bootstrap.csv"
)

print("FINAL TEST RESULTS:")
display(test_summary.round(4))

print("BOOTSTRAP CONFIDENCE INTERVALS:")
display(bootstrap_df.round(4))

In [ ]:
# ============================================================
# CELL 16 — CONSOLIDATED EVALUATION METRICS
# Purpose: Display all evaluation metrics in one cell.
# ============================================================

print("========== VALIDATION METRICS ==========")

display(
    pd.read_csv(
        EV / "val_summary.csv",
        index_col=0
    ).round(4)
)

print("========== TEST METRICS ==========")

display(
    pd.read_csv(
        EV / "test_summary.csv",
        index_col=0
    ).round(4)
)

print("========== CONFIDENCE INTERVALS ==========")

display(
    pd.read_csv(
        EV / "test_bootstrap.csv",
        index_col=0
    ).round(4)
)

for metric in [
    "iou", "dice", "precision", "recall", "f1"
]:
    print(f"PER-CLASS {metric.upper()}")

    display(
        pd.read_csv(
            EV / f"test_per_class_{metric}.csv",
            index_col=0
        ).round(4)
    )

In [ ]:
# ============================================================
# CELL 26 — PLOT STYLE + PRE-FLIGHT CHECK (run once, before the new cells)
# Purpose: Big, readable fonts and colour-blind-safe colours for every
#          figure below; check that earlier cells defined what we need;
#          log environment versions (Step 1 requirement).
# ============================================================
import json, math, shutil, platform
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 14,
    "axes.titlesize": 17,
    "axes.titleweight": "bold",
    "axes.labelsize": 15,
    "axes.labelweight": "bold",
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "legend.frameon": True,
    "figure.titlesize": 20,
    "figure.titleweight": "bold",
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

MODEL_LABELS = {
    "baseline": "YOLOv26s\n(baseline)",
    "dino": "YOLOv26s\n+ DINOv3",
    "byol": "YOLOv26s\n+ BYOL",
    "mae": "YOLOv26s\n+ MAE",
}
MODEL_LABELS_1LINE = {k: v.replace("\n", " ") for k, v in MODEL_LABELS.items()}

# Okabe-Ito palette: distinguishable for colour-blind readers and on print.
MODEL_COLORS = {
    "baseline": "#555555",
    "dino": "#0072B2",
    "byol": "#D55E00",
    "mae": "#009E73",
}
CLASS_COLORS_7 = ["#222222", "#009E73", "#D55E00", "#B59F00",
                  "#0072B2", "#E69F00", "#CC79A7"]   # background + 6 classes

# ---- pre-flight: names that the earlier cells must have defined ----
REQUIRED = ["EV", "FIG", "DS", "RUNS", "CKPT", "LOGS", "EXPS", "BEST_WEIGHTS",
            "MASK_NAMES", "YOLO_NAMES", "NUM_CLASSES", "NC", "SEED", "DEVICE",
            "IMG_SIZE", "man", "FT_EPOCHS", "SSL_EPOCHS"]
missing = [n for n in REQUIRED if n not in globals()]
if missing:
    print("MISSING (run the earlier cells / restore your config cell first):")
    print("  ", missing)
else:
    print("All required variables from earlier cells are present.")

# ---- Step 1: log library and GPU versions ----
try:
    import torch, ultralytics, sklearn
    print("python        :", platform.python_version())
    print("torch         :", torch.__version__)
    print("ultralytics   :", ultralytics.__version__)
    print("scikit-learn  :", sklearn.__version__)
    print("seed          :", SEED if "SEED" in globals() else "(undefined)")
    print("device        :", DEVICE if "DEVICE" in globals() else "(undefined)")
    if torch.cuda.is_available():
        print("GPU           :", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Version logging skipped:", exc)

if "FIG" in globals():
    FIG.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# CELL 27 (REPLACEMENT) — CONFUSION MATRIX, compact "classic" style
# Same look as the reference screenshot: Blues colormap, small integer
# numbers, "Ground truth"/"Predicted" labels, vertical x labels, colour bar
# "Row-normalized %".  Only change: number colour is chosen automatically
# (white on dark cells, black on light cells) by measuring each cell's
# real brightness, so every number is readable.
# Keeps the same name/arguments, so cells 27A-27E keep working unchanged.
# File size: figures are saved to FIG at 150 dpi (full quality for the report)
# and shown inline at 85 dpi, so each output embedded in the notebook is small.
# ============================================================
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

CM_COLOR_BY = "count"      # "count" = exactly like the screenshot | "percent" = colour by row-%
CM_TITLE_NAME = "YOLOv26s"  # title becomes "<name> Confusion Matrix (<model>)"
CM_MODEL_TAG = {"baseline": "", "dino": "DINOv3", "byol": "BYOL", "mae": "MAE"}  # title becomes "<name> Confusion Matrix"
CM_CMAP = LinearSegmentedColormap.from_list(
    "cm_green_yellow", ["#006400", "#0F8A18", "#66DD22", "#FFFF00"])


def _count_text(n):
    n = int(n)
    if n >= 1_000_000:
        return f"{n / 1e6:.1f}M"
    if n >= 10_000:
        return f"{n / 1e3:.0f}K"
    return f"{n:,}"


def plot_confusion_matrix(method, split="test", mode="percent", show=True):
    """Same name/arguments as before, so cells 27A-27E work unchanged.
       mode is kept for compatibility; the picture always shows count + %."""
    saved = np.load(EV / f"{split}_{method}.npz")
    cm_full = saved["cms"].sum(axis=0).astype(float)
    cm = cm_full[1:, 1:]                 # 6 disease classes (drop Background)
    n = cm.shape[0]
    pct = 100 * cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    names = [MASK_NAMES[i].replace("_", " ").upper() for i in range(1, n + 1)]

    data = cm if CM_COLOR_BY == "count" else pct
    vmax = max(data.max(), 1)

    fig, ax = plt.subplots(figsize=(8.5, 7))
    ax.imshow(data, cmap=CM_CMAP, vmin=0, vmax=vmax, aspect="auto")

    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=11)
    ax.set_yticklabels(names, fontsize=11)
    ax.set_xlabel("Predicted", fontsize=12, labelpad=10)
    ax.set_ylabel("True", fontsize=12, labelpad=10)
    ax.set_xticks(np.arange(-.5, n, 1), minor=True)
    ax.set_yticks(np.arange(-.5, n, 1), minor=True)
    ax.grid(which="minor", color="black", linewidth=2)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", length=0)

    for i in range(n):
        for j in range(n):
            dark_text = (data[i, j] / vmax) > 0.22     # bright cell -> black text
            ax.text(j, i, f"{_count_text(cm[i, j])}\n{pct[i, j]:.1f}%",
                    ha="center", va="center", fontsize=13, fontweight="bold",
                    color="black" if dark_text else "white")

        ax.set_title(f"{CM_TITLE_NAME}{f' ({CM_MODEL_TAG[method]})' if CM_MODEL_TAG.get(method) else ''} - Confusion Matrix",
                 fontsize=15, fontweight="bold", pad=10)
    fig.tight_layout()
    out = FIG / f"{method}_confusion_{mode}.png"
    fig.savefig(out, dpi=200)
    if show:
        plt.show()
    plt.close(fig)
    return out


print("plot_confusion_matrix(method) is ready (InceptionV3 style).")

In [ ]:
# CELL 27A — Confusion matrix: YOLO26s (baseline)
plot_confusion_matrix("baseline")


In [ ]:
# CELL 27B — Confusion matrix: YOLO26s + DINOv3
plot_confusion_matrix("dino")


In [ ]:
# CELL 27C — Confusion matrix: YOLO26s + BYOL
plot_confusion_matrix("byol")


In [ ]:
# CELL 27D — Confusion matrix: YOLO26s + MAE
plot_confusion_matrix("mae")


In [ ]:
# CELL 27E — Confusion matrices as raw pixel COUNTS (Step 6 asks for counts too)
#for _m in EXPS:
    #plot_confusion_matrix(_m, mode="counts")


In [ ]:
# Shared history loader for MODEL-WISE and PAIRWISE plots
# All comparisons below use the same supervised validation metric/loss.
def load_finetune_history(method):
    path = RUNS / f"{method}_real_ssl_e{FT_EPOCHS}" / "results.csv"
    history = pd.read_csv(path)
    history.columns = [x.strip() for x in history.columns]
    return history.drop_duplicates("epoch", keep="last").sort_values("epoch")

HISTORIES = {method: load_finetune_history(method) for method in EXPS}
LOSS_KEYS = sorted({col for h in HISTORIES.values() for col in h.columns
                    if (col.startswith("train/") or col.startswith("val/")) and col.endswith("_loss")})
MAP_KEYS = [key for key in ["metrics/mAP50-95(M)", "metrics/mAP50-95(B)"]
            if any(key in h.columns for h in HISTORIES.values())]
print("Supervised loss keys:", LOSS_KEYS)
print("mAP keys:", MAP_KEYS)

In [ ]:

# ONE MODEL PER CELL — YOLO26s + DINOv3
# Train/validation loss and mAP — 3 graphs per row

import math
import matplotlib.pyplot as plt

history = HISTORIES["dino"]
keys = LOSS_KEYS + MAP_KEYS

ncols = 3
nrows = math.ceil(len(keys) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(18, 4.5 * nrows),
    squeeze=False
)

for ax, key in zip(axes.flat, keys):
    if key in history:
        ax.plot(
            history["epoch"],
            history[key],
            marker="o",
            linewidth=2,
            label="YOLOv26s + DINOv3"
        )

    ax.set_title(key, fontsize=12, fontweight="bold")
    ax.set_xlabel("Fine-tuning epoch")
    ax.set_ylabel(key)
    ax.set_xticks(EPOCH_TICKS)
    ax.grid(alpha=0.3)

    if key in history:
        ax.legend(loc="best")

# Remove unused subplots
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)

fig.suptitle("YOLOv26s (DINOv3) - Training Curves", fontsize=15, fontweight="bold")

fig.tight_layout()

fig.savefig(
    FIG / "dino_modelwise_training_curves.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ONE MODEL PER CELL — YOLO26s + BYOL
# Train/validation loss and mAP — 3 graphs per row

import math
import matplotlib.pyplot as plt

history = HISTORIES["byol"]
keys = LOSS_KEYS + MAP_KEYS

ncols = 3
nrows = math.ceil(len(keys) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(18, 4.5 * nrows),
    squeeze=False
)

for ax, key in zip(axes.flat, keys):
    if key in history:
        ax.plot(
            history["epoch"],
            history[key],
            marker="o",
            linewidth=2,
            label="YOLOv26s + BYOL"
        )

    ax.set_title(key, fontsize=12, fontweight="bold")
    ax.set_xlabel("Fine-tuning epoch")
    ax.set_ylabel(key)
    ax.set_xticks(EPOCH_TICKS)
    ax.grid(alpha=0.3)

    if key in history:
        ax.legend(loc="best")

# Hide unused subplots
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)

fig.suptitle("YOLOv26s (BYOL) - Training Curves", fontsize=15, fontweight="bold")

fig.tight_layout()

fig.savefig(
    FIG / "byol_modelwise_training_curves.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ONE MODEL PER CELL — YOLO26s + MAE
# Train/validation loss and mAP — 3 graphs per row

import math
import matplotlib.pyplot as plt

history = HISTORIES["mae"]
keys = LOSS_KEYS + MAP_KEYS

ncols = 3
nrows = math.ceil(len(keys) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(18, 4.5 * nrows),
    squeeze=False
)

for ax, key in zip(axes.flat, keys):
    if key in history:
        ax.plot(
            history["epoch"],
            history[key],
            marker="o",
            linewidth=2,
            label="YOLOv26s + MAE"
        )

    ax.set_title(key, fontsize=12, fontweight="bold")
    ax.set_xlabel("Fine-tuning epoch")
    ax.set_ylabel(key)
    ax.set_xticks(EPOCH_TICKS)
    ax.grid(alpha=0.3)

    if key in history:
        ax.legend(loc="best")

# Hide unused subplots
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)

fig.suptitle("YOLOv26s (MAE) - Training Curves", fontsize=15, fontweight="bold")

fig.tight_layout()

fig.savefig(
    FIG / "mae_modelwise_training_curves.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Reusable comparison functions — each call occurs in its OWN following code cell.
def pairwise_training_plots(method_a, method_b, label_a, label_b):
    ha, hb = HISTORIES[method_a], HISTORIES[method_b]
    keys = LOSS_KEYS + MAP_KEYS
    if not keys:
        raise RuntimeError("Training history contains no comparable loss/mAP columns")
    fig, axes = plt.subplots(len(keys), 1, figsize=(9, max(4, len(keys)*2.8)), squeeze=False)
    for ax, key in zip(axes.flat, keys):
        for h, label in ((ha, label_a), (hb, label_b)):
            if key in h:
                ax.plot(h["epoch"], h[key], marker="o", label=label)
        ax.set(title=key, xlabel="Fine-tuning epoch")
        ax.set_xticks(EPOCH_TICKS)
        ax.grid(alpha=.3)
        ax.legend(loc="best")
    fig.suptitle(f"{label_a} vs {label_b} — supervised curves", y=1.005)
    fig.tight_layout()
    fig.savefig(FIG / f"{method_a}_vs_{method_b}_loss_map.png", dpi=150, bbox_inches="tight")
    plt.show()

    summary = pd.read_csv(EV / "test_summary.csv", index_col=0)
    numeric = summary.loc[[method_a, method_b]].select_dtypes(include=[np.number])
    delta = numeric.loc[method_b] - numeric.loc[method_a]
    diff = pd.DataFrame({label_a: numeric.loc[method_a], label_b: numeric.loc[method_b],
                         f"Delta ({label_b} minus {label_a})": delta})
    display(diff.round(4))
    diff.to_csv(EV / f"{method_a}_vs_{method_b}_metric_differences.csv")
    preferred = [x for x in ["mIoU_7", "mDice_7", "mF1_7", "pixel_accuracy"] if x in numeric.columns]
    if not preferred:
        preferred = list(numeric.columns[:min(5, len(numeric.columns))])
    if preferred:
        fig, ax = plt.subplots(figsize=(9, 4))
        delta.loc[preferred].plot(kind="bar", ax=ax)
        ax.axhline(0, color="black", lw=.8)
        ax.set(title=f"Test metric difference: {label_b} - {label_a}", ylabel="Absolute metric difference")
        ax.tick_params(axis="x", rotation=25)
        ax.grid(axis="y", alpha=.3)
        fig.tight_layout()
        fig.savefig(FIG / f"{method_a}_vs_{method_b}_metric_delta.png", dpi=150)
        plt.show()

In [ ]:

# PAIRWISE DIFFERENCE — YOLO26s vs YOLO26s + MAE
# Train/validation loss and mAP — 3 graphs per row

import math
import matplotlib.pyplot as plt

history_1 = HISTORIES["baseline"]
history_2 = HISTORIES["mae"]

model_1 = "YOLOv26s"
model_2 = "MAE"

keys = LOSS_KEYS + MAP_KEYS

ncols = 3
nrows = math.ceil(len(keys) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(18, 4.5 * nrows),
    squeeze=False
)

for ax, key in zip(axes.flat, keys):

    if key in history_1:
        ax.plot(
            history_1["epoch"],
            history_1[key],
            marker="o",
            linewidth=2,
            label=model_1
        )

    if key in history_2:
        ax.plot(
            history_2["epoch"],
            history_2[key],
            marker="s",
            linewidth=2,
            label=model_2
        )

    ax.set_title(
        key,
        fontsize=11,
        fontweight="bold"
    )

    ax.set_xlabel("Fine-tuning epoch")
    ax.set_ylabel(key)
    ax.set_xticks(EPOCH_TICKS)
    ax.grid(alpha=0.3)
    ax.legend(loc="best")

# Hide unused subplots
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)

fig.suptitle(f"YOLOv26s & ({model_2}) - Training Curves", fontsize=15, fontweight="bold")

fig.tight_layout()

fig.savefig(
    FIG / "pairwise_yolo26_&_mae_training_curves.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# PAIRWISE DIFFERENCE — YOLO26s vs YOLO26s + DINOv3
# Train/validation loss and mAP — 3 graphs per row

import math
import matplotlib.pyplot as plt

history_1 = HISTORIES["baseline"]
history_2 = HISTORIES["dino"]

model_1 = "YOLOv26s"
model_2 = "DINOv3"

keys = LOSS_KEYS + MAP_KEYS

ncols = 3
nrows = math.ceil(len(keys) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(18, 4.5 * nrows),
    squeeze=False
)

for ax, key in zip(axes.flat, keys):

    if key in history_1:
        ax.plot(
            history_1["epoch"],
            history_1[key],
            marker="o",
            linewidth=2,
            label=model_1
        )

    if key in history_2:
        ax.plot(
            history_2["epoch"],
            history_2[key],
            marker="s",
            linewidth=2,
            label=model_2
        )

    ax.set_title(
        key,
        fontsize=11,
        fontweight="bold"
    )

    ax.set_xlabel("Fine-tuning epoch")
    ax.set_ylabel(key)
    ax.set_xticks(EPOCH_TICKS)
    ax.grid(alpha=0.3)

    if key in history_1 or key in history_2:
        ax.legend(loc="best")

# Hide unused subplots
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)

fig.suptitle(f"YOLOv26s & ({model_2}) - Training Curves", fontsize=15, fontweight="bold")

fig.tight_layout()

fig.savefig(
    FIG / "pairwise_yolo26_&_dinov3_training_curves.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# PAIRWISE DIFFERENCE — YOLO26s vs YOLO26s + BYOL
# Train/validation loss and mAP — 3 graphs per row

import math
import matplotlib.pyplot as plt

history_1 = HISTORIES["baseline"]
history_2 = HISTORIES["byol"]

model_1 = "YOLOv26s"
model_2 = "BYOL"

keys = LOSS_KEYS + MAP_KEYS

ncols = 3
nrows = math.ceil(len(keys) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(18, 4.5 * nrows),
    squeeze=False
)

for ax, key in zip(axes.flat, keys):

    if key in history_1:
        ax.plot(
            history_1["epoch"],
            history_1[key],
            marker="o",
            linewidth=2,
            label=model_1
        )

    if key in history_2:
        ax.plot(
            history_2["epoch"],
            history_2[key],
            marker="s",
            linewidth=2,
            label=model_2
        )

    ax.set_title(
        key,
        fontsize=11,
        fontweight="bold"
    )

    ax.set_xlabel("Fine-tuning epoch")
    ax.set_ylabel(key)
    ax.set_xticks(EPOCH_TICKS)
    ax.grid(alpha=0.3)

    if key in history_1 or key in history_2:
        ax.legend(loc="best")

# Hide unused subplots
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)

fig.suptitle(f"YOLOv26s & ({model_2}) - Training Curves", fontsize=15, fontweight="bold")

fig.tight_layout()

fig.savefig(
    FIG / "pairwise_yolo26_&_byol_training_curves.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# PAIRWISE DIFFERENCE — YOLO26s + BYOL vs YOLO26s + MAE
# Train/validation loss and mAP — 3 graphs per row

import math
import matplotlib.pyplot as plt

history_1 = HISTORIES["byol"]
history_2 = HISTORIES["mae"]

model_1 = "BYOL"
model_2 = "MAE"

keys = LOSS_KEYS + MAP_KEYS

ncols = 3
nrows = math.ceil(len(keys) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(18, 4.5 * nrows),
    squeeze=False
)

for ax, key in zip(axes.flat, keys):

    if key in history_1:
        ax.plot(
            history_1["epoch"],
            history_1[key],
            marker="o",
            linewidth=2,
            label=model_1
        )

    if key in history_2:
        ax.plot(
            history_2["epoch"],
            history_2[key],
            marker="s",
            linewidth=2,
            label=model_2
        )

    ax.set_title(
        key,
        fontsize=11,
        fontweight="bold"
    )

    ax.set_xlabel("Fine-tuning epoch")
    ax.set_ylabel(key)
    ax.set_xticks(EPOCH_TICKS)
    ax.grid(alpha=0.3)

    if key in history_1 or key in history_2:
        ax.legend(loc="best")

# Hide unused subplots
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)

fig.suptitle(f"YOLOv26s ({model_1} & {model_2}) - Training Curves", fontsize=15, fontweight="bold")

fig.tight_layout()

fig.savefig(
    FIG / "pairwise_byol_&_mae_training_curves.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()


In [ ]:
# ============================================================
# CELL 18 — QUALITATIVE SEGMENTATION COMPARISON
# Purpose: Show input, ground truth, and four predictions.
# ============================================================

MASK_COLORS = np.array([
    [0, 0, 0],
    [60, 180, 75],
    [230, 25, 75],
    [255, 225, 25],
    [0, 130, 200],
    [245, 130, 48],
    [145, 30, 180]
], dtype=np.uint8)


def color_mask(mask):
    return MASK_COLORS[
        mask.astype(np.uint8)
    ]


test_table = man[
    man["split"] == "test"
]

selected_uids = []

# Select one example from each of the six classes.
for class_name in YOLO_NAMES.values():
    sample = test_table[
        test_table["class"] == class_name
    ]

    if len(sample):
        selected_uids.append(
            str(sample.iloc[0]["uid"])
        )

In [ ]:
# MODEL-WISE qualitative segmentation — YOLO26s + DINOv3, no other models
fig, axes = plt.subplots(len(selected_uids), 3, figsize=(11, 3.5*len(selected_uids)), squeeze=False)
for row, uid in enumerate(selected_uids):
    image_path = next((DS / "images/test").glob(f"{uid}.*"))
    image = np.asarray(Image.open(image_path).convert("RGB"))
    gt = np.asarray(Image.open(DS / "masks/test" / f"{uid}.png"))
    pred = np.asarray(Image.open(EV / "predictions/test/dino" / f"{uid}.png"))
    for col, panel in enumerate([image, color_mask(gt), color_mask(pred)]):
        axes[row,col].imshow(panel)
        axes[row,col].axis("off")
        if row == 0: axes[row,col].set_title(["Original", "Ground truth", "YOLOv26s + DINOv3"][col])
fig.tight_layout()
fig.savefig(FIG / "dino_qualitative.png", dpi=150)
plt.show()

In [ ]:
# MODEL-WISE qualitative segmentation — YOLO26s + BYOL, no other models
fig, axes = plt.subplots(len(selected_uids), 3, figsize=(11, 3.5*len(selected_uids)), squeeze=False)
for row, uid in enumerate(selected_uids):
    image_path = next((DS / "images/test").glob(f"{uid}.*"))
    image = np.asarray(Image.open(image_path).convert("RGB"))
    gt = np.asarray(Image.open(DS / "masks/test" / f"{uid}.png"))
    pred = np.asarray(Image.open(EV / "predictions/test/byol" / f"{uid}.png"))
    for col, panel in enumerate([image, color_mask(gt), color_mask(pred)]):
        axes[row,col].imshow(panel)
        axes[row,col].axis("off")
        if row == 0: axes[row,col].set_title(["Original", "Ground truth", "YOLOv26s + BYOL"][col])
fig.tight_layout()
fig.savefig(FIG / "byol_qualitative.png", dpi=150)
plt.show()

In [ ]:
# MODEL-WISE qualitative segmentation — YOLO26s + MAE, no other models
fig, axes = plt.subplots(len(selected_uids), 3, figsize=(11, 3.5*len(selected_uids)), squeeze=False)
for row, uid in enumerate(selected_uids):
    image_path = next((DS / "images/test").glob(f"{uid}.*"))
    image = np.asarray(Image.open(image_path).convert("RGB"))
    gt = np.asarray(Image.open(DS / "masks/test" / f"{uid}.png"))
    pred = np.asarray(Image.open(EV / "predictions/test/mae" / f"{uid}.png"))
    for col, panel in enumerate([image, color_mask(gt), color_mask(pred)]):
        axes[row,col].imshow(panel)
        axes[row,col].axis("off")
        if row == 0: axes[row,col].set_title(["Original", "Ground truth", "YOLOv26s + MAE"][col])
fig.tight_layout()
fig.savefig(FIG / "mae_qualitative.png", dpi=150)
plt.show()

In [ ]:
# Pairwise qualitative comparison helper; invoked separately per pair below.
def pairwise_masks(a, b, label_a, label_b):
    fig, axes = plt.subplots(len(selected_uids), 4, figsize=(15, 3.5*len(selected_uids)), squeeze=False)
    for row, uid in enumerate(selected_uids):
        im_path = next((DS / "images/test").glob(f"{uid}.*"))
        image = np.asarray(Image.open(im_path).convert("RGB"))
        gt = np.asarray(Image.open(DS / "masks/test" / f"{uid}.png"))
        pred_a = np.asarray(Image.open(EV / "predictions/test" / a / f"{uid}.png"))
        pred_b = np.asarray(Image.open(EV / "predictions/test" / b / f"{uid}.png"))
        for col, panel in enumerate([image, color_mask(gt), color_mask(pred_a), color_mask(pred_b)]):
            axes[row,col].imshow(panel)
            axes[row,col].axis("off")
            if row == 0:
                axes[row,col].set_title(["Original", "Ground truth", label_a, label_b][col])
    fig.tight_layout()
    fig.savefig(FIG / f"{a}_vs_{b}_qualitative.png", dpi=150)
    plt.show()

In [ ]:
# QUALITATIVE PAIRWISE — YOLO26s vs YOLO26s + MAE
pairwise_masks("baseline", "mae", "YOLOv26s", "YOLOv26s + MAE")

In [ ]:
# QUALITATIVE PAIRWISE — YOLO26s vs YOLO26s + DINOv3
pairwise_masks("baseline", "dino", "YOLOv26s", "YOLOv26s + DINOv3")

In [ ]:
# QUALITATIVE PAIRWISE — YOLO26s vs YOLO26s + BYOL
pairwise_masks("baseline", "byol", "YOLOv26s", "YOLOv26s + BYOL")

In [ ]:
# QUALITATIVE PAIRWISE — YOLO26s + BYOL vs YOLO26s + MAE
pairwise_masks("byol", "mae", "YOLOv26s + BYOL", "YOLOv26s + MAE")

In [ ]:

# ============================================================
# CELL 19 — CORRECTED GRAD-CAM FOR YOLO26s
# Purpose:
# 1. Fix the gradient tracking error.
# 2. Generate class-score Grad-CAM heatmaps.
# 3. Compare Baseline, DINO, BYOL, and MAE.
# 4. Save visualizations for the research report.
# ============================================================

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import gc

from pathlib import Path
from PIL import Image
from ultralytics import YOLO


# ------------------------------------------------------------
# STEP 1 — SELECT AN IMAGE AND ITS TARGET CLASS
# ------------------------------------------------------------

TEST_FILES = sorted(
    p for p in (DS / "images/test").iterdir()
    if p.is_file()
)

assert TEST_FILES, "No test images found."

sample_path = TEST_FILES[0]

sample_rgb = np.asarray(
    Image.open(sample_path)
    .convert("RGB")
    .resize((IMG_SIZE, IMG_SIZE))
)

# Load the original semantic ground-truth mask.
gt_cam = np.asarray(
    Image.open(
        DS / "masks/test" / f"{sample_path.stem}.png"
    )
)

if gt_cam.ndim == 3:
    gt_cam = gt_cam[..., 0]

present, counts = np.unique(
    gt_cam[gt_cam > 0],
    return_counts=True
)

TARGET_CLASS = (
    int(present[np.argmax(counts)]) - 1
    if len(present) > 0 else 0
)

# Input tensor, RGB in [0, 1].
x_cam = (
    torch.from_numpy(sample_rgb.copy())
    .permute(2, 0, 1)
    .unsqueeze(0)
    .float()
    .to(DEVICE) / 255.0
)

print("Image:", sample_path.name)
print("Target class:", YOLO_NAMES[TARGET_CLASS])
print("Image tensor:", x_cam.shape)


# ------------------------------------------------------------
# STEP 2 — EXTRACT DIFFERENTIABLE CLASS SCORES
# ------------------------------------------------------------

def extract_class_scores(output):
    """
    Locate raw YOLO class-score tensors.

    Prefer the one-to-many branch when available.
    The scores have expected shape:
        [batch, num_classes, locations]
    """

    if isinstance(output, dict):

        # Prefer a raw prediction branch that is connected
        # to the model's feature maps.
        for branch in ["one2many", "one2one"]:
            if branch in output:
                found = extract_class_scores(output[branch])
                if found is not None:
                    return found

        scores = output.get("scores")

        if isinstance(scores, torch.Tensor):
            if scores.ndim == 3:
                if scores.shape[1] == NC:
                    return scores
                if scores.shape[2] == NC:
                    return scores.transpose(1, 2)

        for value in output.values():
            found = extract_class_scores(value)
            if found is not None:
                return found

    elif isinstance(output, (tuple, list)):
        for item in output:
            found = extract_class_scores(item)
            if found is not None:
                return found

    return None


# ------------------------------------------------------------
# STEP 3 — CORRECTED GRAD-CAM FUNCTION
# ------------------------------------------------------------

def gradcam_yolo(weights_path, image_tensor, class_id):

    # Load the trained model.
    wrapper = YOLO(str(weights_path))

    network = (
        wrapper.model
        .float()
        .to(DEVICE)
        .eval()
    )

    # Critical fix:
    # Enable gradients even if checkpoint loading froze them.
    network.requires_grad_(True)

    # Layer 16 is the selected intermediate feature layer
    # from our original YOLO26s Grad-CAM experiment.
    TARGET_LAYER_INDEX = 16

    assert len(network.model) > TARGET_LAYER_INDEX, (
        "Target feature layer not found."
    )

    target_layer = network.model[TARGET_LAYER_INDEX]

    saved = {}

    # Save only the forward activation.
    # No tensor.register_hook() is needed.
    def activation_hook(module, inputs, output):
        if not isinstance(output, torch.Tensor):
            raise RuntimeError(
                "Selected layer did not return a tensor."
            )

        saved["activation"] = output

    handle = target_layer.register_forward_hook(
        activation_hook
    )

    try:
        # Enable autograd explicitly.
        with torch.inference_mode(False), torch.enable_grad():

            input_tensor = (
                image_tensor.detach()
                .clone()
                .to(DEVICE)
                .requires_grad_(True)
            )

            network.zero_grad(set_to_none=True)

            # Forward pass without YOLO.predict().
            raw_output = network(input_tensor)

            if "activation" not in saved:
                raise RuntimeError(
                    "Feature activation was not captured."
                )

            activation = saved["activation"]

            print(
                "Activation shape:",
                tuple(activation.shape)
            )
            print(
                "Gradient tracking:",
                activation.requires_grad
            )

            if not activation.requires_grad:
                raise RuntimeError(
                    "Gradients are still disabled at layer 16."
                )

            # Extract raw differentiable class scores.
            scores = extract_class_scores(raw_output)

            if scores is None:
                raise RuntimeError(
                    "Raw YOLO26 class scores not found. "
                    "Inspect the model head output."
                )

            if not scores.requires_grad:
                raise RuntimeError(
                    "Class scores are detached from the "
                    "computation graph."
                )

            assert 0 <= class_id < scores.shape[1]

            # Select the desired papaya leaf class.
            class_logits = scores[:, class_id, :]

            # Use the highest-scoring spatial locations.
            top_k = min(20, class_logits.shape[-1])

            target_score = (
                class_logits.sigmoid()
                .topk(top_k, dim=-1)
                .values.mean()
            )

            # Critical fix:
            # Calculate gradients directly with autograd.
            gradients = torch.autograd.grad(
                outputs=target_score,
                inputs=activation,
                retain_graph=False,
                create_graph=False,
                allow_unused=False
            )[0]

            # Global average pooling across spatial positions.
            channel_weights = gradients.mean(
                dim=(2, 3),
                keepdim=True
            )

            # Weighted feature activation.
            cam = F.relu(
                (
                    channel_weights * activation
                ).sum(dim=1, keepdim=True)
            )

            # Resize heatmap to input resolution.
            cam = F.interpolate(
                cam,
                size=image_tensor.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

            cam = (
                cam[0, 0]
                .detach()
                .float()
                .cpu()
                .numpy()
            )

            # Normalize the heatmap to [0, 1].
            cam_min = cam.min()
            cam_max = cam.max()

            if cam_max - cam_min > 1e-8:
                cam = (
                    (cam - cam_min)
                    / (cam_max - cam_min)
                )
            else:
                cam = np.zeros_like(cam)

            return cam

    finally:
        # Remove the hook and release temporary GPU memory.
        handle.remove()
        del network, wrapper
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ------------------------------------------------------------
# STEP 4 — GENERATE GRAD-CAM FOR ALL FOUR MODELS
# ------------------------------------------------------------

EXPS = ["baseline", "dino", "byol", "mae"]

# Semantic ground-truth mask of the same test image (shown instead of the photo).
sample_mask_rgb = color_mask(
    np.asarray(
        Image.fromarray(gt_cam.astype(np.uint8))
        .resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    )
)

fig, axes = plt.subplots(
    1, 5,
    figsize=(21, 5)
)

axes[0].imshow(sample_mask_rgb)
axes[0].set_title("Semantic Mask")
axes[0].axis("off")

gradcam_results = {}

for ax, method in zip(axes[1:], EXPS):

    print("\nGenerating Grad-CAM:", method.upper())

    heatmap = gradcam_yolo(
        BEST_WEIGHTS[method],
        x_cam,
        TARGET_CLASS
    )

    gradcam_results[method] = heatmap

    ax.imshow(sample_mask_rgb)

    ax.imshow(
        heatmap,
        cmap="jet",
        alpha=0.45,
        vmin=0,
        vmax=1
    )

    ax.set_title(method.upper())
    ax.axis("off")

    # Save each model's heatmap separately.
    np.save(
        FIG / f"gradcam_{method}_e{FT_EPOCHS}.npy",
        heatmap
    )

plt.suptitle(
    f"YOLOv26s (All Approaches) - Grad-CAM, {YOLO_NAMES[TARGET_CLASS]}",
    fontsize=15, fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    FIG / f"gradcam_all_models_e{FT_EPOCHS}.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("\nGrad-CAM processing completed.")
print("Saved under:", FIG)

In [ ]:
# ============================================================
# CELL 20 — LIME EXPLAINABILITY
# Purpose: Explain the predicted semantic-mask coverage.
# ============================================================

from lime import lime_image
from skimage.segmentation import (
    slic, mark_boundaries
)

SELECTED_MODEL = json.loads(
    (EV / "selected_model.json").read_text()
)["selected"]

lime_model = YOLO(
    BEST_WEIGHTS[SELECTED_MODEL]
)


def lime_predict(images):
    pil_images = [
        Image.fromarray(
            np.asarray(
                image, dtype=np.uint8
            )
        )
        for image in images
    ]

    predictions = lime_model.predict(
        pil_images,
        imgsz=IMG_SIZE,
        conf=.25,
        iou=.7,
        retina_masks=True,
        device=0,
        batch=8,
        verbose=False
    )

    coverages = []

    for result in predictions:
        semantic = semantic_from_result(
            result,
            result.orig_shape
        )

        coverage = np.bincount(
            semantic.ravel(),
            minlength=NUM_CLASSES
        ).astype(float)

        coverage /= max(
            semantic.size, 1
        )

        coverages.append(coverage)

    return np.asarray(coverages)


def load_lime_image(path):
    return np.asarray(
        Image.open(path).convert("RGB").resize((640, 480))
    )


# LIME needs a class that the model actually predicts. A 5-epoch model
# can return NO mask for an image, which gives zero coverage for every
# class. So use the first of the first 40 test images that has a prediction.
lime_path = TEST_FILES[0]
lime_image_array = load_lime_image(lime_path)
initial_coverage = lime_predict([lime_image_array])[0]

if initial_coverage[1:].sum() == 0:
    for candidate in TEST_FILES[:40]:
        array = load_lime_image(candidate)
        coverage = lime_predict([array])[0]

        if coverage[1:].sum() > 0:
            lime_path = candidate
            lime_image_array = array
            initial_coverage = coverage
            break

if initial_coverage[1:].sum() > 0:
    LIME_CLASS = int(np.argmax(initial_coverage[1:])) + 1
else:
    # No prediction on any of these images: explain the true class
    # (TARGET_CLASS comes from cell 19) and say so.
    LIME_CLASS = int(TARGET_CLASS) + 1
    print("WARNING: model predicted nothing on the first 40 test images; "
          "the LIME map for the ground-truth class will be flat.")

print("LIME image:", lime_path.name)

explainer = lime_image.LimeImageExplainer(
    random_state=SEED
)

explanation = explainer.explain_instance(
    lime_image_array,
    lime_predict,
    labels=(LIME_CLASS,),
    top_labels=None,      # explain exactly LIME_CLASS (default 5 overrides labels)
    num_samples=120,
    batch_size=8,
    hide_color=0,
    segmentation_fn=lambda image: slic(
        image,
        n_segments=55,
        compactness=10,
        sigma=1,
        start_label=0
    )
)

visualization, regions = (
    explanation.get_image_and_mask(
        LIME_CLASS,
        positive_only=True,
        num_features=8,
        hide_rest=False
    )
)

# Semantic ground-truth mask of the explained image (shown instead of the photo).
_lime_gt = np.asarray(Image.open(DS / "masks/test" / f"{lime_path.stem}.png"))
if _lime_gt.ndim == 3:
    _lime_gt = _lime_gt[..., 0]
lime_mask_rgb = color_mask(
    np.asarray(
        Image.fromarray(_lime_gt.astype(np.uint8))
        .resize((640, 480), Image.NEAREST)
    )
)

fig, axes = plt.subplots(
    1, 2, figsize=(12, 5)
)

axes[0].imshow(lime_mask_rgb)
axes[0].set_title("Semantic Mask")
axes[0].axis("off")

axes[1].imshow(
    mark_boundaries(
        lime_mask_rgb,
        regions
    )
)

axes[1].set_title(
    f"LIME — {MASK_NAMES[LIME_CLASS]}"
)
axes[1].axis("off")

plt.tight_layout()

plt.savefig(
    FIG / f"lime_{SELECTED_MODEL}_e{FT_EPOCHS}.png",
    dpi=150
)

plt.show()

print("Selected model:", SELECTED_MODEL)
print("Explained class:", MASK_NAMES[LIME_CLASS])

In [ ]:
# ============================================================
# CELL 21 — COMPUTATIONAL PERFORMANCE ANALYSIS
# Purpose: Measure parameters, model size, latency and memory.
# ============================================================

from thop import profile

compute_rows = []

sample_image = str(TEST_FILES[0])

for method in EXPS:
    print("Profiling:", method)

    model = YOLO(BEST_WEIGHTS[method])

    num_parameters = sum(
        p.numel()
        for p in model.model.parameters()
    )

    model_size_mb = (
        Path(BEST_WEIGHTS[method]).stat().st_size
        / (1024 ** 2)
    )

    try:
        cpu_model = model.model.cpu().eval()

        dummy = torch.zeros(
            1, 3, IMG_SIZE, IMG_SIZE
        )

        macs, _ = profile(
            cpu_model,
            inputs=(dummy,),
            verbose=False
        )

        gmacs = macs / 1e9

    except Exception as error:
        print("MAC profiling issue:", error)
        gmacs = np.nan

    model.model.to(DEVICE)

    # Warm up the GPU.
    for _ in range(3):
        model.predict(
            sample_image,
            imgsz=IMG_SIZE,
            device=0,
            verbose=False
        )

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    latency_values = []

    for _ in range(15):
        result = model.predict(
            sample_image,
            imgsz=IMG_SIZE,
            device=0,
            verbose=False
        )[0]

        latency = sum(
            result.speed.get(key, 0)
            for key in [
                "preprocess",
                "inference",
                "postprocess"
            ]
        )

        latency_values.append(latency)

    torch.cuda.synchronize()

    avg_latency = float(
        np.mean(latency_values)
    )

    peak_memory = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )

    compute_rows.append({
        "model": method,
        "parameters_M": num_parameters / 1e6,
        "model_size_MB": model_size_mb,
        "estimated_GMACs": gmacs,
        "latency_ms": avg_latency,
        "FPS": (
            1000 / avg_latency
            if avg_latency > 0 else np.nan
        ),
        "peak_inference_GPU_MB": peak_memory
    })

    del model
    cleanup_gpu()

compute_df = pd.DataFrame(
    compute_rows
).set_index("model")

compute_df.to_csv(
    EV / "compute_profile.csv"
)

display(compute_df.round(3))

In [ ]:
# COMPUTATION DIFFERENCE — BASELINE vs MAE; separate cell
items = compute_df.loc[["baseline", "mae"]].select_dtypes(include=[np.number])
delta = items.loc["mae"] - items.loc["baseline"]
display(pd.concat([items.T, delta.rename("mae minus baseline")], axis=1).round(3))
fig, ax = plt.subplots(figsize=(9,4))
delta.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=.8)
ax.set(ylabel="Absolute difference (per axis units)")
ax.set_title("YOLOv26s (MAE) vs YOLOv26s - Computation Difference", fontsize=15, fontweight="bold")
ax.grid(axis="y", alpha=.3)
fig.tight_layout()
fig.savefig(FIG / "baseline_vs_mae_compute_delta.png", dpi=150)
plt.show()

In [ ]:
# COMPUTATION DIFFERENCE — BASELINE vs DINO; separate cell
items = compute_df.loc[["baseline", "dino"]].select_dtypes(include=[np.number])
delta = items.loc["dino"] - items.loc["baseline"]
display(pd.concat([items.T, delta.rename("dino minus baseline")], axis=1).round(3))
fig, ax = plt.subplots(figsize=(9,4))
delta.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=.8)
ax.set(ylabel="Absolute difference (per axis units)")
ax.set_title("YOLOv26s (DINOv3) vs YOLOv26s - Computation Difference", fontsize=15, fontweight="bold")
ax.grid(axis="y", alpha=.3)
fig.tight_layout()
fig.savefig(FIG / "baseline_vs_dino_compute_delta.png", dpi=150)
plt.show()

In [ ]:
# COMPUTATION DIFFERENCE — BASELINE vs BYOL; separate cell
items = compute_df.loc[["baseline", "byol"]].select_dtypes(include=[np.number])
delta = items.loc["byol"] - items.loc["baseline"]
display(pd.concat([items.T, delta.rename("byol minus baseline")], axis=1).round(3))
fig, ax = plt.subplots(figsize=(9,4))
delta.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=.8)
ax.set(ylabel="Absolute difference (per axis units)")
ax.set_title("YOLOv26s (BYOL) vs YOLOv26s - Computation Difference", fontsize=15, fontweight="bold")
ax.grid(axis="y", alpha=.3)
fig.tight_layout()
fig.savefig(FIG / "baseline_vs_byol_compute_delta.png", dpi=150)
plt.show()

In [ ]:
# COMPUTATION DIFFERENCE — BYOL vs MAE; separate cell
items = compute_df.loc[["byol", "mae"]].select_dtypes(include=[np.number])
delta = items.loc["mae"] - items.loc["byol"]
display(pd.concat([items.T, delta.rename("mae minus byol")], axis=1).round(3))
fig, ax = plt.subplots(figsize=(9,4))
delta.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=.8)
ax.set(ylabel="Absolute difference (per axis units)")
ax.set_title("YOLOv26s (MAE vs BYOL) - Computation Difference", fontsize=15, fontweight="bold")
ax.grid(axis="y", alpha=.3)
fig.tight_layout()
fig.savefig(FIG / "byol_vs_mae_compute_delta.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 22 — NATIVE YOLO26 INSTANCE SEGMENTATION METRICS
# Purpose: Measure box mAP and mask mAP for all four models.
# ============================================================

map_rows = []

for method in EXPS:
    print("Calculating mAP:", method)

    model = YOLO(
        BEST_WEIGHTS[method]
    )

    result = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=FT_BATCH,
        conf=.001,
        iou=.7,
        device=0,
        plots=False,
        verbose=False
    )

    map_rows.append({
        "model": method,
        "mask_mAP50": float(
            result.seg.map50
        ),
        "mask_mAP50_95": float(
            result.seg.map
        ),
        "box_mAP50": float(
            result.box.map50
        ),
        "box_mAP50_95": float(
            result.box.map
        )
    })

    del model
    cleanup_gpu()

map_df = pd.DataFrame(
    map_rows
).set_index("model")

map_df.to_csv(
    EV / "test_ultralytics_map.csv"
)

display(map_df.round(4))

print("YOLO instance-segmentation metrics completed.")

In [ ]:
# ============================================================
# CELL 23 — FINAL CONSOLIDATED MODEL RESULTS
# Purpose: Present all metrics together in one table.
# ============================================================

semantic_results = pd.read_csv(
    EV / "test_summary.csv",
    index_col=0
)

bootstrap_results = pd.read_csv(
    EV / "test_bootstrap.csv",
    index_col=0
)

map_results = pd.read_csv(
    EV / "test_ultralytics_map.csv",
    index_col=0
)

compute_results = pd.read_csv(
    EV / "compute_profile.csv",
    index_col=0
)

FINAL_RESULTS = (
    semantic_results
    .join(bootstrap_results)
    .join(map_results)
    .join(compute_results)
)

FINAL_RESULTS.to_csv(
    EV / f"final_results_e{FT_EPOCHS}.csv"
)

print("=" * 70)
print(f"YOLO26s {FT_EPOCHS}-EPOCH COMPARATIVE RESULTS")
print("=" * 70)

display(FINAL_RESULTS.round(4))


# Self-supervised training summary.
ssl_rows = []

for method in ["dino", "byol", "mae"]:
    path = (
        LOGS
        / f"{method}_yolo26s_{TAG}_history.csv"
    )

    history = pd.read_csv(path)

    loss_column = (
        "loss"
    )

    ssl_rows.append({
        "method": method,
        "epochs": len(history),
        "loss_type": loss_column,
        "final_SSL_loss": float(
            history[loss_column].iloc[-1]
        ),
        "minimum_SSL_loss": float(
            history[loss_column].min()
        )
    })

SSL_SUMMARY = pd.DataFrame(
    ssl_rows
)

print("\nSELF-SUPERVISED PRETRAINING SUMMARY")
display(SSL_SUMMARY.round(4))

assert (
    SSL_SUMMARY["epochs"] == SSL_EPOCHS
).all()

selection = json.loads(
    (EV / "selected_model.json").read_text()
)

print("\nValidation-selected model:")
print(selection["selected"])

print("\nHighest observed test mIoU:")
print(
    FINAL_RESULTS["mIoU_7"].idxmax()
)

print("\nAll results saved successfully.")

In [ ]:
# ============================================================
# CELL 24 — FINAL INFERENCE PIPELINE
# Purpose: Show input image, predicted mask, and overlay.
# ============================================================

SELECTED_MODEL = json.loads(
    (EV / "selected_model.json").read_text()
)["selected"]

final_model = YOLO(
    BEST_WEIGHTS[SELECTED_MODEL]
)

# Use one example from the test dataset.
# Replace with another image path when needed.
IMAGE_TO_PREDICT = str(TEST_FILES[0])

original = np.asarray(
    Image.open(
        IMAGE_TO_PREDICT
    ).convert("RGB")
)

prediction = final_model.predict(
    IMAGE_TO_PREDICT,
    imgsz=IMG_SIZE,
    conf=.25,
    iou=.7,
    retina_masks=True,
    device=0,
    verbose=False
)[0]

semantic_prediction = semantic_from_result(
    prediction,
    original.shape[:2]
)

colored_prediction = color_mask(
    semantic_prediction
)

overlay = cv2.addWeighted(
    original, .6,
    colored_prediction, .4,
    0
)

fig, axes = plt.subplots(
    1, 3, figsize=(16, 5)
)

titles = [
    "Original Papaya Leaf",
    "Predicted Semantic Mask",
    "Segmentation Overlay"
]

images = [
    original,
    colored_prediction,
    overlay
]

for ax, image, title in zip(
    axes, images, titles
):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()

plt.savefig(
    FIG / f"final_inference_e{FT_EPOCHS}.png",
    dpi=150
)

plt.show()

Image.fromarray(
    semantic_prediction
).save(
    EV / f"final_predicted_mask_e{FT_EPOCHS}.png"
)

print("Selected model:", SELECTED_MODEL)
print("Training epochs:", FT_EPOCHS)
print("Inference completed.")

In [ ]:
# ============================================================
# CELL 28 — MCC, MACRO-F1, 1,000-RESAMPLE BOOTSTRAP, VERDICTS, test_metrics.json
# Purpose: Close Step 5 + Step 6 gaps: multiclass MCC, Macro-F1, 1,000
#          image-level bootstrap CIs for mIoU / Macro-F1 / MCC, paired
#          differences vs baseline, a better/same/worse verdict, and the
#          per-class stability table. (Old cell 15 used 500 resamples, mIoU only.)
# Uses the per-image confusion matrices already saved by cell 15.
# ============================================================
N_BOOT = 1000


def mcc_from_cm(cm):
    cm = cm.astype(np.float64)
    s, c = cm.sum(), np.trace(cm)
    t, p = cm.sum(axis=1), cm.sum(axis=0)
    denom = np.sqrt((s ** 2 - (p ** 2).sum()) * (s ** 2 - (t ** 2).sum()))
    return float((c * s - (p * t).sum()) / denom) if denom > 0 else 0.0


def scores_from_cm(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = np.where(tp + fp + fn > 0, tp / (tp + fp + fn), np.nan)
        f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), np.nan)
    return dict(mIoU_7=np.nanmean(iou), MacroF1_7=np.nanmean(f1),
                MCC=mcc_from_cm(cm), iou_per_class=iou)


per_image = {m: np.load(EV / f"test_{m}.npz") for m in EXPS}
_uids = per_image["baseline"]["uids"]
assert all(np.array_equal(per_image[m]["uids"], _uids) for m in EXPS)
n_img = len(_uids)

rng = np.random.default_rng(SEED)
boot_idx = rng.integers(0, n_img, size=(N_BOOT, n_img))      # SAME resamples for all models

boot = {m: {"mIoU_7": [], "MacroF1_7": [], "MCC": [], "iou_pc": []} for m in EXPS}
for m in EXPS:
    cms = per_image[m]["cms"]
    for idx in boot_idx:
        sc = scores_from_cm(cms[idx].sum(axis=0))
        boot[m]["mIoU_7"].append(sc["mIoU_7"])
        boot[m]["MacroF1_7"].append(sc["MacroF1_7"])
        boot[m]["MCC"].append(sc["MCC"])
        boot[m]["iou_pc"].append(sc["iou_per_class"])
    for k in boot[m]:
        boot[m][k] = np.asarray(boot[m][k])

VERDICT_METRIC = "mIoU_7"
rows, test_metrics = [], {}
for m in EXPS:
    point = scores_from_cm(per_image[m]["cms"].sum(axis=0))
    entry = {"model": m}
    for k in ["mIoU_7", "MacroF1_7", "MCC"]:
        lo, hi = np.percentile(boot[m][k], [2.5, 97.5])
        d = boot[m][k] - boot["baseline"][k]
        dlo, dhi = np.percentile(d, [2.5, 97.5])
        entry.update({k: point[k], f"{k}_CI_low": lo, f"{k}_CI_high": hi,
                      f"{k}_diff": point[k] - scores_from_cm(
                          per_image['baseline']['cms'].sum(axis=0))[k],
                      f"{k}_diff_low": dlo, f"{k}_diff_high": dhi})
    if m == "baseline":
        entry["verdict_vs_baseline"] = "reference"
    elif entry[f"{VERDICT_METRIC}_diff_low"] > 0:
        entry["verdict_vs_baseline"] = "BETTER"
    elif entry[f"{VERDICT_METRIC}_diff_high"] < 0:
        entry["verdict_vs_baseline"] = "WORSE"
    else:
        entry["verdict_vs_baseline"] = "SAME (CI includes 0)"
    rows.append(entry)
    test_metrics[m] = {k: (float(v) if isinstance(v, (np.floating, float)) else v)
                       for k, v in entry.items()}

boot_df = pd.DataFrame(rows).set_index("model")
boot_df.to_csv(EV / "test_bootstrap_1000.csv")

(EV / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))

show_cols = ["mIoU_7", "mIoU_7_CI_low", "mIoU_7_CI_high", "MacroF1_7", "MCC",
             "mIoU_7_diff", "mIoU_7_diff_low", "mIoU_7_diff_high", "verdict_vs_baseline"]
display(boot_df[show_cols].round(4))

# ---- per-class stability table (mean +- std of IoU over bootstraps) ----
stab = {}
for m in EXPS:
    mu = np.nanmean(boot[m]["iou_pc"], axis=0)
    sd = np.nanstd(boot[m]["iou_pc"], axis=0)
    stab[m] = [f"{a:.3f} ± {b:.3f}" for a, b in zip(mu, sd)]
stability_df = pd.DataFrame(stab, index=[MASK_NAMES[i] for i in range(NUM_CLASSES)])
stability_df.to_csv(EV / "per_class_stability.csv")
print("Per-class IoU stability (mean ± std over 1,000 bootstrap resamples):")
display(stability_df)

# ---- forest plot of paired differences vs baseline ----
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5), sharey=True)
others = [m for m in EXPS if m != "baseline"]
for ax, k in zip(axes, ["mIoU_7", "MacroF1_7", "MCC"]):
    for i, m in enumerate(others):
        d, lo, hi = (test_metrics[m][f"{k}_diff"], test_metrics[m][f"{k}_diff_low"],
                     test_metrics[m][f"{k}_diff_high"])
        ax.errorbar(d, i, xerr=[[d - lo], [hi - d]], fmt="o", ms=13, capsize=8,
                    elinewidth=3.5, color=MODEL_COLORS[m])
        ax.text(hi + 0.005, i + 0.18, f"{d:+.3f}", fontsize=14, fontweight="bold",
                color=MODEL_COLORS[m])
    ax.margins(x=0.35)
    ax.axvline(0, color="black", lw=2, ls="--")
    ax.set_yticks(range(len(others)))
    ax.set_yticklabels([MODEL_LABELS_1LINE[m] for m in others], fontsize=14)
    ax.set_title(f"Δ {k} vs baseline (95% CI)")
    ax.set_xlabel("← worse than baseline        better →")
fig.suptitle("YOLOv26s (SSL) & YOLOv26s - Paired Bootstrap Differences", fontsize=15, fontweight="bold", y=1.03)
fig.tight_layout()
fig.savefig(FIG / "paired_bootstrap_forest.png")
plt.show()
print("Saved:", EV / "test_metrics.json")


In [ ]:
# ============================================================
# CELL 29 — METRIC GRAPHS: mIoU, Accuracy, Precision, Recall (+ F1)
# Purpose: Clear bar charts with value labels; mIoU has 95% bootstrap CI.
# Reads: test_summary.csv, test_bootstrap.csv (cell 15) and test_bootstrap_1000.csv (cell 28)
# ============================================================
test_summary_df = pd.read_csv(EV / "test_summary.csv", index_col=0).loc[EXPS]
_ci_path = EV / "test_bootstrap_1000.csv"            # from CELL 30 if it was run
if not _ci_path.exists():
    _ci_path = EV / "test_bootstrap.csv"              # fallback: original cell 15
ci_df = pd.read_csv(_ci_path, index_col=0).loc[EXPS]
ci_df = ci_df.rename(columns={"mIoU_7_CI_low": "CI_low", "mIoU_7_CI_high": "CI_high"})

def _bar_panel(ax, column, title, ci=None, ylabel=None):
    vals = test_summary_df[column].values.astype(float)
    colours = [MODEL_COLORS[m] for m in EXPS]
    yerr = None
    if ci is not None:
        yerr = np.vstack([vals - ci["CI_low"].values, ci["CI_high"].values - vals])
        yerr = np.clip(yerr, 0, None)
    bars = ax.bar(range(len(EXPS)), vals, color=colours, edgecolor="black",
                  linewidth=1.2, width=0.68, yerr=yerr,
                  error_kw=dict(ecolor="black", capsize=7, elinewidth=2))
    top = max(vals.max() * 1.18, 0.05)
    ax.set_ylim(0, min(top, 1.12) if vals.max() < 0.9 else 1.12)
    for i, (b, v) in enumerate(zip(bars, vals)):
        top_of_bar = v + (yerr[1][i] if yerr is not None else 0)
        ax.text(b.get_x() + b.get_width() / 2, top_of_bar + 0.012, f"{v:.3f}",
                ha="center", va="bottom", fontsize=15, fontweight="bold")
    ax.set_xticks(range(len(EXPS)))
    ax.set_xticklabels([MODEL_LABELS[m] for m in EXPS], fontsize=13)
    ax.set_title(title)
    ax.set_ylabel(ylabel or column)
    ax.grid(axis="x", visible=False)

PANELS = [
    ("mIoU_7", "Mean IoU (7 classes) with 95% CI", ci_df),
    ("pixel_accuracy", "Pixel accuracy", None),
    ("mPrecision_fg6", "Mean precision (6 disease/leaf classes)", None),
    ("mRecall_fg6", "Mean recall (6 disease/leaf classes)", None),
]
fig, axes = plt.subplots(2, 2, figsize=(16, 11.5))
for ax, (col, title, ci) in zip(axes.flat, PANELS):
    _bar_panel(ax, col, title, ci)
fig.suptitle("YOLOv26s (All Approaches) - Test-Set Metrics", fontsize=15, fontweight="bold", y=1.0)
fig.tight_layout()
fig.savefig(FIG / "metrics_miou_acc_precision_recall.png")
plt.show()

# each metric also saved as its own PNG for the report
for col, title, ci in PANELS + [("mF1_fg6", "Mean F1 (6 classes)", None),
                                ("mDice_7", "Mean Dice (7 classes)", None)]:
    f, a = plt.subplots(figsize=(8.5, 6))
    _bar_panel(a, col, title, ci)
    f.tight_layout()
    f.savefig(FIG / f"metric_{col}.png")
    plt.close(f)
print("Saved under:", FIG)


In [ ]:
# ============================================================
# CELL 30 — PER-CLASS IoU / PRECISION / RECALL (grouped bars)
# Purpose: Show which disease classes each model gets right or wrong.
# Reads: test_per_class_{iou,precision,recall}.csv from cell 15
# ============================================================
def plot_per_class(metric, ax):
    table = pd.read_csv(EV / f"test_per_class_{metric}.csv", index_col=0)[EXPS]
    classes = [str(c).replace("_", " ").title() for c in table.index]
    x = np.arange(len(classes))
    width = 0.2
    for k, m in enumerate(EXPS):
        ax.bar(x + (k - 1.5) * width, table[m].fillna(0).values, width,
               label=MODEL_LABELS_1LINE[m], color=MODEL_COLORS[m],
               edgecolor="black", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(classes, fontsize=14)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel(metric.upper() if metric == "iou" else metric.title())
    ax.set_title(f"Per-class {metric.upper() if metric == 'iou' else metric}")
    ax.grid(axis="x", visible=False)

fig, axes = plt.subplots(3, 1, figsize=(17, 17), sharex=False)
for ax, metric in zip(axes, ["iou", "precision", "recall"]):
    plot_per_class(metric, ax)
axes[0].legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.28), fontsize=14)
fig.suptitle("YOLOv26s (All Approaches) - Per-Class Test Metrics", fontsize=15, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "per_class_iou_precision_recall.png")
plt.show()


In [ ]:
# ============================================================
# CELL 31 (REPLACEMENT) — AUC: functions + per-model cache (NO plots here)
# Purpose: pixel-level PR-AUC / ROC-AUC, computed ONE MODEL AT A TIME.
# NOTE: scores come from instance-mask confidences (YOLO has no per-pixel
#       softmax), so AUC here is an approximation. State this in the report.
# Test split only; checkpoints are not changed.
# Results are cached in AUC_RESULTS (and the slow score pass is cached on
# disk), so re-running any plot cell is fast.
# ============================================================
from sklearn.metrics import (precision_recall_curve, roc_curve,
                             average_precision_score, auc as sk_auc)
import cv2, gc

AUC_SIZE = 160          # pixels are scored on a 160x160 grid to keep memory low


def score_maps_from_result(result, shape):
    """(7, H, W) scores: class c = max confidence of instances of class c
       covering the pixel; background = 1 - max foreground score."""
    h, w = shape
    scores = np.zeros((NUM_CLASSES, h, w), dtype=np.float32)
    if result.masks is not None:
        masks = result.masks.data.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        conf = result.boxes.conf.cpu().numpy()
        for m, c, p in zip(masks, classes, conf):
            if m.shape != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            region = m > .5
            scores[c + 1][region] = np.maximum(scores[c + 1][region], p)
    scores[0] = 1.0 - scores[1:].max(axis=0)
    return scores


def collect_pixel_scores(method, split="test"):
    cache = EV / f"{split}_{method}_pixel_scores.npz"
    if cache.exists():
        z = np.load(cache)
        return z["y"], z["s"]
    model = YOLO(BEST_WEIGHTS[method])
    paths = sorted(p for p in (DS / "images" / split).iterdir() if p.is_file())
    cfg = {**PRED_CONFIG, "conf": 0.001}          # low threshold -> full score range
    ys, ss = [], []
    for start in tqdm(range(0, len(paths), 8), desc=f"scores {method}"):
        batch = paths[start:start + 8]
        for path, result in zip(batch, model.predict([str(p) for p in batch],
                                                     batch=len(batch), **cfg)):
            gt = np.asarray(Image.open(DS / "masks" / split / f"{path.stem}.png"))
            if gt.ndim == 3:
                gt = gt[..., 0]
            sc = score_maps_from_result(result, gt.shape)
            sc = np.stack([cv2.resize(c, (AUC_SIZE, AUC_SIZE),
                                      interpolation=cv2.INTER_AREA) for c in sc], -1)
            gt = cv2.resize(gt.astype(np.uint8), (AUC_SIZE, AUC_SIZE),
                            interpolation=cv2.INTER_NEAREST)
            ys.append(gt.ravel())
            ss.append(sc.reshape(-1, NUM_CLASSES).astype(np.float16))
    y, s = np.concatenate(ys), np.concatenate(ss)
    np.savez_compressed(cache, y=y, s=s)
    del model
    cleanup_gpu()
    return y, s


def auc_tables(y, s):
    """One-vs-rest curves + AUCs for every class."""
    out = {}
    for k in range(NUM_CLASSES):
        t = (y == k).astype(int)
        if t.sum() == 0 or t.sum() == len(t):
            out[k] = None
            continue
        sc = s[:, k].astype(np.float32)
        p, r, _ = precision_recall_curve(t, sc)
        fpr, tpr, _ = roc_curve(t, sc)
        out[k] = dict(pr=(r, p), roc=(fpr, tpr),
                      ap=average_precision_score(t, sc), roc_auc=sk_auc(fpr, tpr))
    return out


if "AUC_RESULTS" not in globals():
    AUC_RESULTS = {}


def get_auc(method):
    """Compute (once) and return the AUC tables for ONE model."""
    if method not in AUC_RESULTS:
        y, s = collect_pixel_scores(method)
        AUC_RESULTS[method] = auc_tables(y, s)
        del y, s
        gc.collect()
    return AUC_RESULTS[method]


AUC_SHOW_DPI = 80       # inline preview resolution (keeps notebook small)
AUC_SAVE_DPI = 150      # saved PNG resolution


APPROACH = {"baseline": "", "dino": "DINOv3", "byol": "BYOL", "mae": "MAE"}


def plot_auc_curve(method, kind):
    """ONE figure for ONE model.  kind = 'pr' or 'roc'."""
    res = get_auc(method)
    fig, ax = plt.subplots(figsize=(8.2, 7.6))
    for k, v in res.items():
        if v is None:
            continue
        x, yv = v["pr"] if kind == "pr" else v["roc"]
        val = v["ap"] if kind == "pr" else v["roc_auc"]
        ax.plot(x, yv, lw=2.6, color=CLASS_COLORS_7[k],
                label=f"{MASK_NAMES[k].replace('_', ' ').title()}  ({val:.2f})")
    if kind == "roc":
        ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.5)
    ax.set_xlabel("Recall" if kind == "pr" else "False positive rate")
    ax.set_ylabel("Precision" if kind == "pr" else "True positive rate")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_title(f"YOLOv26s{f' ({APPROACH[method]})' if APPROACH[method] else ''} - "
                 + ("Precision Recall Curve" if kind == "pr" else "ROC Curve"), fontsize=15, fontweight="bold")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, fontsize=12)
    fig.tight_layout()
    out = FIG / f"auc_{kind}_{method}.png"
    fig.savefig(out, dpi=AUC_SAVE_DPI, bbox_inches="tight")
    fig.set_dpi(AUC_SHOW_DPI)
    plt.show()
    plt.close(fig)
    return out


def plot_auc_model(method):
    """PR curves, then ROC curves, for ONE model only (two separate figures)."""
    plot_auc_curve(method, "pr")
    plot_auc_curve(method, "roc")


print("AUC helpers ready. Next: run one cell per model (31A-31D), then 31E.")


In [ ]:
# CELL 31A — AUC curves: YOLO26s (baseline) only
plot_auc_model("baseline")

In [ ]:
# CELL 31B — AUC curves: YOLO26s + DINOv3 only
plot_auc_model("dino")

In [ ]:
# CELL 31C — AUC curves: YOLO26s + BYOL only
plot_auc_model("byol")

In [ ]:
# CELL 31D — AUC curves: YOLO26s + MAE only
plot_auc_model("mae")

In [ ]:
# CELL 31E — AUC summary table + bars (writes test_auc_summary.csv, needed by CELL 38)
rows = []
for _m in EXPS:
    res = get_auc(_m)                                   # cached if already computed
    valid = [v for v in res.values() if v is not None]
    fg = [res[k] for k in range(1, NUM_CLASSES) if res[k]]
    rows.append({"model": _m,
                 "macro_PR_AUC_7": np.mean([v["ap"] for v in valid]),
                 "macro_ROC_AUC_7": np.mean([v["roc_auc"] for v in valid]),
                 "macro_PR_AUC_fg6": np.mean([v["ap"] for v in fg]),
                 "macro_ROC_AUC_fg6": np.mean([v["roc_auc"] for v in fg])})
auc_df = pd.DataFrame(rows).set_index("model")
auc_df.to_csv(EV / "test_auc_summary.csv")
display(auc_df.round(4))

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(EXPS)); w = 0.36
b1 = ax.bar(x - w / 2, auc_df.loc[EXPS, "macro_PR_AUC_7"], w, label="Macro PR-AUC",
            color="#0072B2", edgecolor="black")
b2 = ax.bar(x + w / 2, auc_df.loc[EXPS, "macro_ROC_AUC_7"], w, label="Macro ROC-AUC",
            color="#E69F00", edgecolor="black")
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                f"{b.get_height():.3f}", ha="center", fontsize=13, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([MODEL_LABELS[m] for m in EXPS])
ax.set_ylim(0, 1.12); ax.set_title("YOLOv26s (All Approaches) - AUC", fontsize=15, fontweight="bold")
ax.legend(loc="lower right"); ax.grid(axis="x", visible=False)
fig.tight_layout(); fig.savefig(FIG / "auc_summary_bars.png", dpi=150)
fig.set_dpi(80); plt.show(); plt.close(fig)


In [ ]:
# ============================================================
# CELL 32 — FIXED 12-IMAGE TEST REFERENCE SET + PREDICTION OVERLAYS (Step 2 / Step 5)
# Purpose: Lock a constant list of 12 test images (2 per disease class) and
#          draw the same 12 for every model so runs are comparable.
# Saves: reference_12.json, <method>_reference12_overlay.png
# ============================================================
REF_PATH = EV / "reference_12.json"
_class_col = "class" if "class" in man.columns else "label"

if REF_PATH.exists():
    REFERENCE_12 = json.loads(REF_PATH.read_text())
else:
    _test = man[man["split"] == "test"].copy()
    _test["uid"] = _test["uid"].astype(str)
    REFERENCE_12 = []
    for cname in YOLO_NAMES.values():                      # 6 classes
        _rows = _test[_test[_class_col].astype(str) == cname].sort_values("uid")
        REFERENCE_12 += [{"uid": u, "class": cname} for u in _rows["uid"].head(2)]
    assert len(REFERENCE_12) == 12, f"Expected 12, got {len(REFERENCE_12)}"
    REF_PATH.write_text(json.dumps(REFERENCE_12, indent=2))
print("Reference set locked:", [r["uid"] for r in REFERENCE_12])


def _find_image(split, uid):
    for p in (DS / "images" / split).iterdir():
        if p.stem == uid:
            return p
    raise FileNotFoundError(uid)


def plot_reference_overlays(method, alpha=0.5):
    model = YOLO(BEST_WEIGHTS[method])
    fig, axes = plt.subplots(3, 4, figsize=(20, 16))
    for ax, ref in zip(axes.flat, REFERENCE_12):
        path = _find_image("test", ref["uid"])
        rgb = np.asarray(Image.open(path).convert("RGB"))
        res = model.predict(str(path), imgsz=IMG_SIZE, conf=.25, iou=.7,
                            retina_masks=True, device=0, verbose=False)[0]
        pred = semantic_from_result(res, rgb.shape[:2])
        gt = np.asarray(Image.open(DS / "masks/test" / f"{ref['uid']}.png"))
        gt = gt[..., 0] if gt.ndim == 3 else gt
        inter = ((pred == gt) & (gt > 0)).sum()
        union = ((pred > 0) | (gt > 0)).sum()
        colour = color_mask(pred)
        overlay = np.where((pred > 0)[..., None],
                           (rgb * (1 - alpha) + colour * alpha).astype(np.uint8), rgb)
        ax.imshow(overlay)
        ax.set_title(f"{ref['class']}\nforeground IoU {inter / max(union, 1):.2f}",
                     fontsize=16)
        ax.axis("off")
    handles = [Patch(color=MASK_COLORS[i] / 255, label=MASK_NAMES[i].title())
               for i in range(1, NUM_CLASSES)]
    fig.legend(handles=handles, loc="lower center", ncol=6, fontsize=15,
               bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"YOLOv26s{ {'baseline': '', 'dino': ' (DINOv3)', 'byol': ' (BYOL)', 'mae': ' (MAE)'}[method]} - Test Reference Set Predictions", fontsize=15, fontweight="bold", y=0.99)
    fig.tight_layout(rect=(0, 0.03, 1, 0.97))
    fig.savefig(FIG / f"{method}_reference12_overlay.png")
    plt.show()
    del model
    cleanup_gpu()


for _m in EXPS:
    plot_reference_overlays(_m)


In [ ]:
# ============================================================
# CELL 33 — GRAD-CAM GRID (improved visibility)
# Purpose: Same Grad-CAM method as cell 19 (reuses gradcam_yolo), but the
#          heat-map is drawn only where importance is high, so the leaf stays
#          visible; shared colour bar; one row per class; big fonts.
# Requires cell 19 (gradcam_yolo) and cell 32 (REFERENCE_12).
# ============================================================
GRADCAM_ROWS_PER_FIGURE = 3          # 6 classes -> 2 figures
_cmap = plt.get_cmap("turbo")


def overlay_cam(ax, rgb, cam, gamma=0.8, max_alpha=0.8):
    ax.imshow(rgb)
    rgba = _cmap(cam)
    rgba[..., 3] = (np.clip(cam, 0, 1) ** gamma) * max_alpha   # low values transparent
    ax.imshow(rgba)
    ax.axis("off")


_first_of_class = {}
for _r in REFERENCE_12:
    _first_of_class.setdefault(_r["class"], _r)
GRADCAM_SAMPLES = list(_first_of_class.values())            # one image per class
_name_to_yolo_id = {v: k for k, v in YOLO_NAMES.items()}

GRADCAM_CACHE = {}
for _chunk_start in range(0, len(GRADCAM_SAMPLES), GRADCAM_ROWS_PER_FIGURE):
    chunk = GRADCAM_SAMPLES[_chunk_start:_chunk_start + GRADCAM_ROWS_PER_FIGURE]
    fig, axes = plt.subplots(len(chunk), 5, figsize=(24, 4.9 * len(chunk)),
                             squeeze=False)
    for r, ref in enumerate(chunk):
        path = _find_image("test", ref["uid"])
        rgb = np.asarray(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
        _gt = np.asarray(Image.open(DS / "masks" / "test" / f"{ref['uid']}.png"))
        if _gt.ndim == 3:
            _gt = _gt[..., 0]
        mask_rgb = color_mask(np.asarray(
            Image.fromarray(_gt.astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)))
        x = (torch.from_numpy(rgb.copy()).permute(2, 0, 1).unsqueeze(0).float()
             .to(DEVICE) / 255.0)
        cid = _name_to_yolo_id[ref["class"]]
        axes[r, 0].imshow(mask_rgb); axes[r, 0].axis("off")
        axes[r, 0].set_title(f"Semantic mask — {ref['class']}", fontsize=13)
        for c, m in enumerate(EXPS, start=1):
            cam = gradcam_yolo(BEST_WEIGHTS[m], x, cid)
            GRADCAM_CACHE[(ref["uid"], m)] = cam
            overlay_cam(axes[r, c], mask_rgb, cam)
            if r == 0:
                axes[r, c].set_title(MODEL_LABELS_1LINE[m], fontsize=13)
    sm = plt.cm.ScalarMappable(cmap=_cmap, norm=plt.Normalize(0, 1))
    cbar = fig.colorbar(sm, ax=axes, fraction=0.015, pad=0.01)
    cbar.set_label("Grad-CAM importance   (low → high)", fontsize=17)
    cbar.ax.tick_params(labelsize=14)
    fig.suptitle("YOLOv26s (All Approaches) - Grad-CAM", fontsize=15, fontweight="bold", y=1.0)
    out = FIG / f"gradcam_grid_{_chunk_start // GRADCAM_ROWS_PER_FIGURE + 1}.png"
    fig.savefig(out)
    plt.show()
    print("Saved:", out)
np.savez_compressed(EV / "gradcam_maps.npz",
                    **{f"{u}__{m}": v for (u, m), v in GRADCAM_CACHE.items()})

In [ ]:
# ============================================================
# CELL 34 — LIME PANELS (improved visibility)
# Purpose: For the selected model, show (1) original, (2) the superpixels LIME
#          says SUPPORT the class (green fill + thick yellow outline), and
#          (3) a signed weight map (red = supports, blue = opposes) with colour bar.
# Requires cell 20 (lime_predict, semantic_from_result) and cell 32.
# Images where the model predicts nothing are skipped (LIME has nothing to explain).
# ============================================================
from lime import lime_image
from skimage.segmentation import slic, mark_boundaries

LIME_SAMPLES = 300
LIME_PANELS = 3                      # how many panels to draw


def lime_panel(method, ref):
    """Draw one LIME panel. Returns False if the model predicts nothing."""
    global lime_model                                  # lime_predict (cell 20) reads this
    lime_model = YOLO(BEST_WEIGHTS[method])
    path = _find_image("test", ref["uid"])
    image = np.asarray(Image.open(path).convert("RGB").resize((640, 480)))
    _gt = np.asarray(Image.open(DS / "masks" / "test" / f"{ref['uid']}.png"))
    if _gt.ndim == 3:
        _gt = _gt[..., 0]
    mask_img = color_mask(np.asarray(
        Image.fromarray(_gt.astype(np.uint8)).resize((640, 480), Image.NEAREST)))

    cover = lime_predict([image])[0]
    if cover[1:].sum() == 0:
        print(f"Skipped {ref['uid']} ({ref['class']}): "
              f"{method} predicted no mask, so LIME has nothing to explain.")
        return False
    label = int(np.argmax(cover[1:])) + 1              # most-predicted class

    explainer = lime_image.LimeImageExplainer(random_state=SEED)
    exp = explainer.explain_instance(
        image, lime_predict, labels=(label,), top_labels=None,   # explain exactly `label`
        num_samples=LIME_SAMPLES, batch_size=8, hide_color=0,
        segmentation_fn=lambda im: slic(im, n_segments=55, compactness=10,
                                        sigma=1, start_label=0))
    segs = exp.segments
    weights = dict(exp.local_exp[label])
    wmap = np.zeros(segs.shape, dtype=np.float32)
    for sid, w in weights.items():
        wmap[segs == sid] = w
    top = sorted(weights.items(), key=lambda kv: kv[1], reverse=True)[:8]
    support = np.isin(segs, [s for s, w in top if w > 0])

    fig, axes = plt.subplots(1, 3, figsize=(24, 7.2))
    axes[0].imshow(mask_img)
    axes[0].set_title(f"Semantic mask — {ref['class']}", fontsize=13)

    tint = mask_img.astype(np.float32).copy()
    tint[support] = 0.55 * tint[support] + 0.45 * np.array([0, 230, 60])
    outlined = mark_boundaries(tint.astype(np.uint8), support.astype(int),
                               color=(1, 0.9, 0), mode="thick")
    axes[1].imshow(outlined)
    axes[1].set_title(f"LIME: top supporting regions\nexplained class = "
                      f"{MASK_NAMES[label].title()}", fontsize=13)

    lim = max(np.abs(wmap).max(), 1e-6)
    axes[2].imshow(mask_img, alpha=0.35)
    im = axes[2].imshow(wmap, cmap="coolwarm", vmin=-lim, vmax=lim, alpha=0.8)
    axes[2].set_title("Superpixel weights\n(red = supports class, blue = opposes)",
                      fontsize=13)
    cbar = fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.02)
    cbar.set_label("LIME weight", fontsize=16)
    cbar.ax.tick_params(labelsize=13)
    for a in axes:
        a.axis("off")
    fig.suptitle(f"YOLOv26s{ {'baseline': '', 'dino': ' (DINOv3)', 'byol': ' (BYOL)', 'mae': ' (MAE)'}[method]} - LIME Explanation", fontsize=15, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(FIG / f"lime_panel_{method}_{ref['uid']}.png")
    plt.show()
    cleanup_gpu()
    return True


_sel = json.loads((EV / "selected_model.json").read_text())["selected"]

# walk the 12 reference images; draw up to LIME_PANELS panels, one per class
drawn, seen_classes = 0, set()
for _ref in REFERENCE_12:
    if drawn >= LIME_PANELS:
        break
    if _ref["class"] in seen_classes:
        continue
    if lime_panel(_sel, _ref):
        drawn += 1
        seen_classes.add(_ref["class"])

if drawn == 0:
    print("The selected model predicted no mask on any reference image. "
          "Train longer (real run) before using LIME.")

In [ ]:
# ============================================================
# CELL 35 — EMBEDDING UTILITIES (shared by cells 36 and 37)
# Purpose: Extract encoder features (global-average-pooled last backbone
#          layer) and score them: KNN probe, PCA, k-means, collapse stats.
# ============================================================
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

_class_col = "class" if "class" in man.columns else "label"
_name_to_idx = {n: i for i, n in YOLO_NAMES.items()}          # 0..5


def split_items(split):
    """Sorted (path, class-index 0..5) list for one split."""
    rows = man[man["split"] == split].copy()
    rows["uid"] = rows["uid"].astype(str)
    lab = dict(zip(rows["uid"], rows[_class_col].astype(str)))
    paths = sorted(p for p in (DS / "images" / split).iterdir() if p.is_file())
    return paths, np.array([_name_to_idx[lab[p.stem]] for p in paths])


@torch.no_grad()
def extract_embeddings(full_model, paths, size=None, batch=16):
    size = size or IMG_SIZE
    backbone = YoloBackbone(full_model).to(DEVICE).eval()
    feats = []
    for i in range(0, len(paths), batch):
        imgs = [np.asarray(Image.open(p).convert("RGB").resize((size, size)))
                for p in paths[i:i + batch]]
        x = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float().to(DEVICE) / 255.
        f = backbone(x)[0].mean(dim=(2, 3))
        feats.append(f.float().cpu().numpy())
    del backbone
    cleanup_gpu()
    return np.concatenate(feats)


def embedding_report(E, y, n_clusters=6, seed=42):
    """PCA(2) + k-means vs true classes. Pure numpy/sklearn."""
    Z = PCA(n_components=2, random_state=seed).fit_transform(E)
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=seed).fit(E)
    return {"pca2": Z, "kmeans": km.labels_,
            "ARI": adjusted_rand_score(y, km.labels_),
            "NMI": normalized_mutual_info_score(y, km.labels_),
            "silhouette": float(silhouette_score(E, y))}


def collapse_stats(E):
    """Representation-collapse indicators (Step 3 health check)."""
    En = E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)
    std_dim = float(En.std(axis=0).mean())                 # ~0 => collapsed
    sv = np.linalg.svd(En - En.mean(0), compute_uv=False)
    p = sv ** 2 / max((sv ** 2).sum(), 1e-12)
    eff_rank = float(np.exp(-(p * np.log(p + 1e-12)).sum()))
    return {"mean_feature_std": std_dim, "effective_rank": eff_rank,
            "ideal_std_1_over_sqrt_D": float(1 / np.sqrt(E.shape[1]))}


def knn_probe(Etr, ytr, Eva, yva, k=20):
    clf = KNeighborsClassifier(n_neighbors=min(k, len(Etr)), metric="cosine")
    clf.fit(Etr, ytr)
    return float((clf.predict(Eva) == yva).mean())

print("Embedding utilities ready.")


In [ ]:
# ============================================================
# CELL 36 — STEP 3 HEALTH CHECKS FOR THE SSL ENCODERS + ssl_weights.pt
# Purpose: KNN probe on FROZEN features (fit on train, scored on val),
#          collapse indicators, loss trend; save ssl_weights.pt per method.
# Uses train and val images only (test split is never touched here).
# ============================================================
SSL_CKPTS = {"dino": DINO_FINAL, "byol": BYOL_FINAL, "mae": MAE_FINAL}
train_paths, y_train = split_items("train")
val_paths, y_val = split_items("val")

health_rows = []
for method in ["random_init"] + list(SSL_CKPTS):
    full = new_full_model()
    if method != "random_init":
        ck = torch.load(SSL_CKPTS[method], map_location="cpu", weights_only=False)
        transfer_backbone(ck["backbone"], full)
        # Artifact required by Step 3: ssl_weights.pt (one per method)
        out_dir = CKPT / method
        out_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(SSL_CKPTS[method], out_dir / "ssl_weights.pt")
    Etr = extract_embeddings(full, train_paths)
    Eva = extract_embeddings(full, val_paths)
    row = {"encoder": method, "KNN_val_acc": knn_probe(Etr, y_train, Eva, y_val)}
    row.update(collapse_stats(Etr))
    if method != "random_init":
        h = pd.read_csv(LOGS / f"{method}_yolo26s_{TAG}_history.csv")
        row["SSL_loss_first"] = float(h["loss"].iloc[0])
        row["SSL_loss_last"] = float(h["loss"].iloc[-1])
        row["SSL_loss_trend"] = "decreasing" if h["loss"].iloc[-1] < h["loss"].iloc[0] \
            else "NOT decreasing"
    health_rows.append(row)
    del full
    cleanup_gpu()

health_df = pd.DataFrame(health_rows).set_index("encoder")
health_df.to_csv(EV / "ssl_health_checks.csv")
display(health_df.round(4))

chance = 1 / len(np.unique(y_train))
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
colors = ["#555555", MODEL_COLORS["dino"], MODEL_COLORS["byol"], MODEL_COLORS["mae"]]
b = axes[0].bar(health_df.index, health_df["KNN_val_acc"], color=colors,
                edgecolor="black", linewidth=1.2)
axes[0].axhline(chance, color="red", ls="--", lw=2, label=f"chance = {chance:.2f}")
for bar in b:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + .012,
                 f"{bar.get_height():.3f}", ha="center", fontsize=15, fontweight="bold")
axes[0].set_ylim(0, 1.1); axes[0].set_title("KNN probe accuracy (frozen features, val)")
axes[0].legend(); axes[0].grid(axis="x", visible=False)
b = axes[1].bar(health_df.index, health_df["effective_rank"], color=colors,
                edgecolor="black", linewidth=1.2)
for bar in b:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                 f"{bar.get_height():.1f}", ha="center", fontsize=15, fontweight="bold")
axes[1].set_title("Effective rank of features (low = collapse risk)")
axes[1].grid(axis="x", visible=False)
fig.tight_layout(); fig.savefig(FIG / "ssl_health_checks.png"); plt.show()


In [ ]:
# ============================================================
# CELL 37 — TEST EMBEDDINGS (embeddings.npy) + PCA / K-MEANS PLOTS (Step 5 / 6)
# Purpose: Save the fine-tuned encoder's test embeddings and draw PCA and
#          k-means clustering plots with readable colours.
# Test images are only passed through the frozen best.pt (no selection).
# ============================================================
test_paths, y_test = split_items("test")
CLASS_PLOT_COLORS = CLASS_COLORS_7[1:]                       # six disease/leaf classes
class_names_6 = [YOLO_NAMES[i].replace("_", " ").title() for i in range(NC)]

# Panel titles: "YOLO26" for the baseline, "YOLO26 (DINOv3)" etc. for SSL models
EMB_TITLE = {"baseline": "YOLOv26s", "dino": "YOLOv26s (DINOv3)",
             "byol": "YOLOv26s (BYOL)", "mae": "YOLOv26s (MAE)"}

EMB, REP = {}, {}
for m in EXPS:
    wrapper = YOLO(BEST_WEIGHTS[m])
    EMB[m] = extract_embeddings(wrapper.model.float(), test_paths)
    np.save(EV / f"embeddings_{m}.npy", EMB[m])
    REP[m] = embedding_report(EMB[m], y_test, n_clusters=NC, seed=SEED)
    del wrapper
    cleanup_gpu()

np.save(EV / "embeddings.npy", EMB[_sel] if "_sel" in globals() else EMB["baseline"])
cluster_df = pd.DataFrame({m: {k: REP[m][k] for k in ["ARI", "NMI", "silhouette"]}
                           for m in EXPS}).T
cluster_df.to_csv(EV / "embedding_cluster_scores.csv")
display(cluster_df.round(4))

fig, axes = plt.subplots(2, 4, figsize=(28, 13))
for c, m in enumerate(EXPS):
    Z = REP[m]["pca2"]
    for k in range(NC):
        sel = y_test == k
        axes[0, c].scatter(Z[sel, 0], Z[sel, 1], s=70, color=CLASS_PLOT_COLORS[k],
                           edgecolor="black", linewidth=0.5, label=class_names_6[k])
    axes[0, c].set_title(EMB_TITLE[m])
    axes[1, c].scatter(Z[:, 0], Z[:, 1], s=70, c=REP[m]["kmeans"], cmap="tab10",
                       edgecolor="black", linewidth=0.5)
    axes[1, c].set_title(f"k-means (k={NC})\nARI {REP[m]['ARI']:.2f} | "
                         f"NMI {REP[m]['NMI']:.2f}")
    for r in range(2):
        axes[r, c].set_xlabel("PC 1"); axes[r, c].set_ylabel("PC 2")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=6, fontsize=16, markerscale=2.2,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle("YOLO26 (All Approaches) - Embeddings PCA and k-Means",
             fontsize=15, fontweight="bold", y=1.0)
fig.tight_layout(rect=(0, 0.04, 1, 0.98))
fig.savefig(FIG / "embeddings_pca_kmeans.png")
plt.show()

In [ ]:
# ============================================================
# CELL 38 — COMPLETE COMPARISON MATRIX (Step 6)
# Purpose: One table: 3 methods + baseline x mIoU, Macro-F1, MCC, PR-AUC,
#          accuracy, precision, recall, parameters, GFLOPs, latency/FPS, verdict.
# Needs: cells 28 and 31, and cell 21 (compute_profile.csv).
# ============================================================
final = pd.read_csv(EV / "test_summary.csv", index_col=0).loc[EXPS][
    ["pixel_accuracy", "mPrecision_fg6", "mRecall_fg6"]]
bt = pd.read_csv(EV / "test_bootstrap_1000.csv", index_col=0).loc[EXPS][
    ["mIoU_7", "mIoU_7_CI_low", "mIoU_7_CI_high", "MacroF1_7", "MacroF1_7_CI_low",
     "MacroF1_7_CI_high", "MCC", "MCC_CI_low", "MCC_CI_high", "verdict_vs_baseline"]]
auc_t = pd.read_csv(EV / "test_auc_summary.csv", index_col=0).loc[EXPS][
    ["macro_PR_AUC_7", "macro_ROC_AUC_7"]]
comp = pd.read_csv(EV / "compute_profile.csv", index_col=0).loc[EXPS]
comp = comp.rename(columns={"estimated_GMACs": "GMACs"})
comp["GFLOPs"] = 2 * comp["GMACs"]            # 1 MAC = 2 FLOPs

COMPLETE = bt.join(final).join(auc_t).join(comp)
COMPLETE.to_csv(EV / "final_results_complete.csv")
display(COMPLETE.round(4))

# Compact heat-table figure for the report
cols = ["mIoU_7", "MacroF1_7", "MCC", "macro_PR_AUC_7", "pixel_accuracy",
        "mPrecision_fg6", "mRecall_fg6"]
fig, ax = plt.subplots(figsize=(15, 5.5))
data = COMPLETE[cols].values.astype(float)
im = ax.imshow(data, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(["mIoU", "Macro-F1", "MCC", "PR-AUC", "Accuracy",
                    "Precision", "Recall"], fontsize=15)
ax.set_yticks(range(len(EXPS)))
ax.set_yticklabels([MODEL_LABELS_1LINE[m] for m in EXPS], fontsize=15)
for i in range(len(EXPS)):
    for j in range(len(cols)):
        ax.text(j, i, f"{data[i, j]:.3f}", ha="center", va="center", fontsize=17,
                fontweight="bold", color="white" if data[i, j] > 0.6 else "black")
ax.grid(False)
fig.colorbar(im, ax=ax, label="score")
ax.set_title("YOLOv26s (All Approaches) - Comparison Matrix", fontsize=15, fontweight="bold")
fig.tight_layout(); fig.savefig(FIG / "comparison_matrix.png"); plt.show()


In [ ]:
# ============================================================
# CELL 39 — STEP 8: GRADIO PROTOTYPE (Upload -> Predict mask -> Explain -> Result)
# Purpose: Interactive app using the validation-selected model.
# NOT TESTED on Kaggle by the author of this cell: run it once and check.
# ============================================================
import subprocess, sys
# Gradio 5.x only: the newest Gradio (6.x) needs huggingface_hub>=1.16, but cell 1
# pins huggingface_hub<1.0 for transformers 4.x. Keep both pins in ONE pip call so
# pip cannot upgrade huggingface_hub underneath the running kernel.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gradio>=5,<6", "huggingface_hub>=0.34,<1.0"], check=True)
# drop any half-imported gradio from an earlier failed attempt
for _name in [m for m in sys.modules
              if m.split(".")[0] in ("gradio", "gradio_client")]:
    del sys.modules[_name]
import gradio as gr
print("gradio", gr.__version__)
from lime import lime_image
from skimage.segmentation import slic, mark_boundaries

APP_METHOD = json.loads((EV / "selected_model.json").read_text())["selected"]
APP_MODEL = YOLO(BEST_WEIGHTS[APP_METHOD])
lime_model = APP_MODEL                                     # used by lime_predict (cell 20)
_cam_cmap = plt.get_cmap("turbo")


def app_predict(image):
    rgb = np.asarray(Image.fromarray(image).convert("RGB"))
    res = APP_MODEL.predict(rgb, imgsz=IMG_SIZE, conf=.25, iou=.7,
                            retina_masks=True, device=0, verbose=False)[0]
    sem = semantic_from_result(res, rgb.shape[:2])
    return rgb, sem, color_mask(sem)


def app_explain(rgb, sem):
    small = np.asarray(Image.fromarray(rgb).resize((IMG_SIZE, IMG_SIZE)))
    lime_in = np.asarray(Image.fromarray(rgb).resize((640, 480)))

    counts = np.bincount(sem.ravel(), minlength=NUM_CLASSES)[1:]
    if counts.sum() == 0:
        # Nothing predicted: there is no class to explain (LIME would crash).
        return small, lime_in, None
    cls0 = int(np.argmax(counts))                                   # YOLO id 0..5

    x = torch.from_numpy(small.copy()).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE) / 255.
    cam = gradcam_yolo(BEST_WEIGHTS[APP_METHOD], x, cls0)
    rgba = _cam_cmap(cam); rgba[..., 3] = (cam ** 0.8) * 0.8
    base = small.astype(np.float32) / 255
    cam_img = (base * (1 - rgba[..., 3:]) + rgba[..., :3] * rgba[..., 3:])
    cam_img = (np.clip(cam_img, 0, 1) * 255).astype(np.uint8)

    exp = lime_image.LimeImageExplainer(random_state=SEED).explain_instance(
        lime_in, lime_predict, labels=(cls0 + 1,), top_labels=None,
        num_samples=150, batch_size=8, hide_color=0,
        segmentation_fn=lambda im: slic(im, n_segments=55, compactness=10,
                                        sigma=1, start_label=0))
    vis, reg = exp.get_image_and_mask(cls0 + 1, positive_only=True, num_features=8,
                                      hide_rest=False)
    lime_img = (mark_boundaries(vis, reg, color=(1, .9, 0), mode="thick") * 255)
    return cam_img, lime_img.astype(np.uint8), cls0


def app_result(rgb, sem, colored):
    overlay = np.where((sem > 0)[..., None],
                       (rgb * 0.5 + colored * 0.5).astype(np.uint8), rgb)
    total = max((sem > 0).sum(), 1)
    lines = []
    for k in range(1, NUM_CLASSES):
        share = (sem == k).sum() / total * 100
        if share > 0.5:
            lines.append(f"{MASK_NAMES[k].title()}: {share:.1f}% of detected leaf area")
    return overlay, "\n".join(lines) or "No leaf / disease region detected."


def run_pipeline(image):
    if image is None:
        raise gr.Error("Upload a leaf photo first.")
    rgb, sem, colored = app_predict(image)
    cam_img, lime_img, _ = app_explain(rgb, sem)
    overlay, summary = app_result(rgb, sem, colored)
    return colored, cam_img, lime_img, overlay, summary


with gr.Blocks(title="Papaya leaf segmentation") as demo:
    gr.Markdown(f"## Papaya leaf disease segmentation — model: **{APP_METHOD}**")
    with gr.Row():
        inp = gr.Image(label="1 · Upload leaf photo", type="numpy")
        out_mask = gr.Image(label="2 · Predicted mask")
    with gr.Row():
        out_cam = gr.Image(label="3a · Explain: Grad-CAM")
        out_lime = gr.Image(label="3b · Explain: LIME")
    with gr.Row():
        out_over = gr.Image(label="4 · Result overlay")
        out_txt = gr.Textbox(label="4 · Detected classes", lines=6)
    gr.Button("Run", variant="primary").click(
        run_pipeline, inp, [out_mask, out_cam, out_lime, out_over, out_txt])

demo.launch(share=True, debug=False)

In [ ]:
# ============================================================
# CELL 25 — FINAL RESULTS BACKUP
# Purpose: Package checkpoints, CSVs, XAI plots and figures.
# ============================================================

BACKUP = (
    W / f"papaya_yolo26_real_dinov3_mae_e{FT_EPOCHS}_results.zip"
)

with zipfile.ZipFile(
    BACKUP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for folder in [
        CKPT,
        LOGS,
        EV
    ]:
        for path in folder.rglob("*"):
            if path.is_file():
                archive.write(
                    path,
                    path.relative_to(EXP_ROOT)
                )

    # Include fine-tuned model checkpoints.
    for method in EXPS:
        run_dir = (
            RUNS / f"{method}_real_ssl_e{FT_EPOCHS}"
        )

        for filename in [
            "weights/best.pt",
            "weights/last.pt",
            "results.csv",
            "init_check.json",
            "DONE.json"
        ]:
            path = run_dir / filename

            if path.exists():
                archive.write(
                    path,
                    path.relative_to(EXP_ROOT)
                )

    # Include experiment configuration.
    config_path = (
        EXP_ROOT / "ft_config.json"
    )

    if config_path.exists():
        archive.write(
            config_path,
            config_path.relative_to(EXP_ROOT)
        )

    # Include dataset configuration and class mapping.
    for filename in [
        "data.yaml",
        "class_map.json",
        "split_table.csv"
    ]:
        path = DS / filename

        if path.exists():
            archive.write(
                path,
                Path("dataset_metadata") / filename
            )

print("Backup created:", BACKUP)

print(
    "Backup size (MB):",
    round(
        BACKUP.stat().st_size / 1024**2,
        2
    )
)

print(f"{FT_EPOCHS}-epoch pipeline completed.")